# 압록강 서바이벌 — 그래픽 자동 생성 (112장)

**처음 한 번**
1. OpenAI API 키(`sk-...`)를 준비합니다. 왼쪽 열쇠 아이콘(보안 비밀)에 이름 `OPENAI_API_KEY`로 넣어 두면 매번 붙여넣지 않아도 됩니다(없으면 실행 중 입력칸이 뜹니다).
2. 런타임 → 모두 실행. Google Drive 연결을 허용하면 그림이 Drive의 `amnok_art` 폴더에 저장되어 런타임이 끊겨도 남습니다.

**사용 순서 (권장)**
1. `LIMIT = 1`로 한 장만 만들어 결과와 미리보기를 확인합니다.
2. 괜찮으면 `LIMIT = 0`(제한 없음)으로 바꾸고 다시 모두 실행합니다. 이미 만든 그림은 건너뜁니다.
3. 끝나면 `game_assets.zip`이 내려받아집니다. 압축을 풀면 `public/assets/...` 구조이므로 GitHub 저장소 루트에 그대로 올리면 게임이 자동으로 사용합니다.

**변환 규칙이 바뀌었을 때(노트북 새 버전)**: `CONVERT_ONLY`를 체크하고 모두 실행하면, Drive에 있는 원본을 다시 생성하지 않고 변환만 새로 합니다.

**마음에 안 드는 그림 다시 만들기**: `REGENERATE`에 항목 이름을 쉼표로 적고 실행합니다(예: `ally_A01,enemy_E04`). 프롬프트가 바뀐 항목만 다시 만들려면 `changed`.

**단계**: `P1`(스테이지 1, 23장) → `P2`(스테이지 1~5) → `P3`(나머지). 필요한 것부터 만들어 비용을 나눠 쓸 수 있습니다.


In [ ]:
#@title 1) 설정 { display-mode: "form" }
#@markdown 체크하면 새로 만들지 않고, 이미 만든 원본(Drive의 raw)을 최신 변환 규칙으로 다시 변환만 함(API 키·비용 불필요)
CONVERT_ONLY = False  #@param {type:"boolean"}
PHASE = "P1"  #@param ["P1", "P2", "P3", "ALL"]
#@markdown 특정 항목만 만들 때(쉼표 구분, 예: ally_A01,enemy_E01). 비우면 PHASE 전체
ONLY = ""  #@param {type:"string"}
#@markdown 이번 실행에서 새로 만들 최대 장수. 처음엔 1로 시험, 0이면 제한 없음
LIMIT = 1  #@param {type:"integer"}
QUALITY = "medium"  #@param ["low", "medium", "high"]
#@markdown auto: 계정에서 쓸 수 있는 최신 이미지 모델을 자동 선택
MODEL = "auto"  #@param ["auto", "gpt-image-2.5-flare", "gpt-image-2.5-sunburst", "gpt-image-1.5", "gpt-image-1"]
#@markdown 동시 요청 수. 429(속도 제한)가 자주 나면 1~2로
CONCURRENCY = 3  #@param {type:"integer"}
#@markdown 전쟁 소재가 과하게 차단되면 low (지원하지 않는 모델이면 자동으로 빠짐)
MODERATION = "auto"  #@param ["auto", "low"]
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
#@markdown 다시 만들 항목(쉼표 구분) 또는 changed(프롬프트가 바뀐 항목 전부)
REGENERATE = ""  #@param {type:"string"}
#@markdown 원본 이미지 zip도 함께 내려받기
ZIP_RAW = False  #@param {type:"boolean"}


In [ ]:
#@title 2) 준비 (Drive·API 키) { display-mode: "form" }
import base64, getpass, hashlib, json, os, random, re, threading, time, zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

BASE = "/content/amnok_art" if IN_COLAB else os.path.abspath("amnok_art")
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/amnok_art"
RAW_DIR = os.path.join(BASE, "raw")
OUT_DIR = os.path.join(BASE, "out")
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
print("저장 위치:", BASE)

def _get_key():
    if os.environ.get("OPENAI_API_KEY"):
        return os.environ["OPENAI_API_KEY"].strip()
    if IN_COLAB:
        try:
            from google.colab import userdata
            k = userdata.get("OPENAI_API_KEY")
            if k:
                return k.strip()
        except Exception:
            pass
    return getpass.getpass("OpenAI API 키를 붙여넣고 Enter: ").strip()

API_KEY = "" if CONVERT_ONLY else _get_key()
if API_KEY:
    os.environ["OPENAI_API_KEY"] = API_KEY  # 다시 실행해도 묻지 않게(이 런타임 안에서만)
    if not API_KEY.startswith("sk-"):
        print("경고: API 키 형식이 sk-로 시작하지 않습니다. 확인해 주세요.")
HEADERS = {"Authorization": "Bearer " + API_KEY}


In [ ]:
#@title 3) 프롬프트 데이터 (건드리지 마세요) { display-mode: "form" }
PROMPTS = json.loads(r'''[
{
"name": "tiles_1",
"phase": "P1",
"size": "1024x1024",
"transparent": false,
"prompt": "Create ONE image containing FOUR seamless ground textures for a retro 2D top-down survival game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 8x8 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCANVAS: 1:1 square split into a 2x2 grid of four equal square tiles, edge to edge, no gaps or borders. Each tile is a 64x64-pixel texture enlarged 8 times, viewed straight down from above (90 degrees), and wraps seamlessly on its own: its left edge continues its right edge and its top edge continues its bottom edge.\nLOOK: flat ground only, medium-dark values and LOW contrast so bright character sprites stand out on top; no outlines on the ground texture itself, no large objects, no strong shadows, no people, no blood; 4-6 small scattered details per tile (pebbles, tufts, cracks, puddles) spread evenly with no obvious repeat. Reading order: top-left, top-right, bottom-left, bottom-right.\nTILES:\n1. T01: riverbank sand and fine gravel beside a wide river (Gyeongsang)\n2. T02: rice-paddy ridges and dry field soil\n3. T03: flat harvested rice paddy with short green-gold stubble (Jeolla)\n4. T04: ochre-red clay road with shallow wheel ruts",
"post": {
"kind": "tiles",
"ids": [
"T01",
"T02",
"T03",
"T04"
]
},
"no": 1
},
{
"name": "ally_A01",
"phase": "P1",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A01 ROK Army rifleman: olive fatigues, steel M1 helmet, M1 Garand rifle, canteen and ammo belt. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A01",
"faction": "ally",
"tag": "S"
},
"no": 2
},
{
"name": "ally_A02",
"phase": "P1",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A02 ROK Army guard trooper: helmet, chest plate, tall riveted steel shield plate held in front, M1 carbine. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A02",
"faction": "ally",
"tag": "S"
},
"no": 3
},
{
"name": "ally_A03",
"phase": "P1",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A03 ROK Army mortar man: helmet, ammo satchel, small 60mm mortar tube carried on the shoulder. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A03",
"faction": "ally",
"tag": "S"
},
"no": 4
},
{
"name": "enemy_E01",
"phase": "P1",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E01 KPA rifleman: khaki-brown uniform, side cap with a red star, bolt-action rifle, charging. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E01",
"faction": "enemy",
"tag": "S"
},
"no": 5
},
{
"name": "enemy_E02",
"phase": "P1",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E02 KPA scout: light gear, crouched fast runner, pistol. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E02",
"faction": "enemy",
"tag": "S"
},
"no": 6
},
{
"name": "enemy_E03",
"phase": "P1",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E03 KPA mortar crewman: mortar tube and shell box. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E03",
"faction": "enemy",
"tag": "S"
},
"no": 7
},
{
"name": "enemy_E04",
"phase": "P1",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E04 T-34 medium tank: olive-green hull, red star on the turret. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: vehicle, aircraft or emplacement: about 85% of the cell width (aircraft fly level, their lowest point on the baseline). Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame loop: treads rolling or wheels turning with exhaust flicker; aircraft: a gentle flight wobble with engine glow\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 hatch closes or crew ducks; 3 armor plates flash steel-blue; 4 hold with a faint shimmer; 5 reopen\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E04",
"faction": "enemy",
"tag": "L"
},
"no": 8
},
{
"name": "enemy_E05",
"phase": "P1",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E05 river-crossing commander: officer cap, medals, binoculars, pistol, wet boots. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: boss: bulkier and taller, about 80% of the cell height, medals and long coat. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E05",
"faction": "enemy",
"tag": "B"
},
"no": 9
},
{
"name": "npc_N01",
"phase": "P1",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of a friendly NPC for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Kind, warm expressions; no injury or distress. A speech bubble may contain three dots only.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: N01 war correspondent: brimmed hat with a press card, trench coat, camera, notebook.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Use only the top 3 rows (each row is one action animated in 5 frames, left to right); rows 4 and 5 stay completely empty. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 15 used cells, as if traced from one model sheet. Side view facing RIGHT toward the player. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 idle (5 frames): gentle breathing, small weight shift, a blink on frame 4, a tiny prop movement (cloth, strap, tool)\nROW 2 talk (5 frames): nod and head tilt, one hand gesturing, mouth opening and closing, a small three-dot speech bubble on frames 2-4\nROW 3 give (5 frames): 1 notice the player; 2 reach into bag or pocket; 3 extend an arm holding a small item with a white-gold glint; 4 item sparkles; 5 pull back with a small friendly wave",
"post": {
"kind": "unit",
"id": "N01",
"faction": "npc",
"tag": "S"
},
"no": 10
},
{
"name": "npc_N02",
"phase": "P1",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of a friendly NPC for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Kind, warm expressions; no injury or distress. A speech bubble may contain three dots only.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: N02 field surgeon: white apron over fatigues, helmet with a red cross, medical bag.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Use only the top 3 rows (each row is one action animated in 5 frames, left to right); rows 4 and 5 stay completely empty. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 15 used cells, as if traced from one model sheet. Side view facing RIGHT toward the player. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 idle (5 frames): gentle breathing, small weight shift, a blink on frame 4, a tiny prop movement (cloth, strap, tool)\nROW 2 talk (5 frames): nod and head tilt, one hand gesturing, mouth opening and closing, a small three-dot speech bubble on frames 2-4\nROW 3 give (5 frames): 1 notice the player; 2 reach into bag or pocket; 3 extend an arm holding a small item with a white-gold glint; 4 item sparkles; 5 pull back with a small friendly wave",
"post": {
"kind": "unit",
"id": "N02",
"faction": "npc",
"tag": "S"
},
"no": 11
},
{
"name": "card_frames",
"phase": "P1",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE image with FIVE card frames for a retro 2D card-and-survival game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nNo skulls, no blood. BACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nCANVAS: 1:1 square; five card frames side by side in one horizontal row, evenly spaced with clear transparent gaps; each card is a 5:7 portrait rectangle (like a 120x168-pixel card enlarged), pixel-stepped rounded corners. Identical layout for all five, all fields left EMPTY: a name banner across the top (top 12% of the card); a small round slot at the top-left corner for a branch icon; a large SQUARE illustration window right under the banner, about 87% of the card width, filled with flat dark gray; a parchment-colored description box below it; three small gem slots along the bottom edge.\nLEFT TO RIGHT: 1 COMMON: olive-green riveted steel frame, 1 gem lit. 2 RARE: silver-blue riveted plates, 2 gems lit. 3 LEGENDARY: brass-gold ornate corners, 3 gems lit, a 50%-opacity golden pixel glow around the card. 4 MINUS: dark maroon frame, yellow-black hazard stripes on the border, barbed wire on the edges, a caution triangle in the icon slot, no gems lit. 5 BACK: card back (also used for face-down cards): dark olive ammo-crate wood pattern, a large pixel star emblem, rope-like border.",
"post": {
"kind": "row",
"count": 5,
"names": [
"frame_common",
"frame_rare",
"frame_legend",
"frame_minus",
"card_back"
],
"folder": "cards",
"w": 120,
"h": 168
},
"no": 12
},
{
"name": "card_tags",
"phase": "P1",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE image with NINE game icons for a retro 2D card-and-survival game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 10x10 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nCANVAS: 1:1 square split into an invisible 3x3 grid; one icon per cell, centered, each designed as a bold 32x32-pixel icon enlarged about 10 times, thick outline, simple shape readable at small size. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nREADING ORDER: 1 rifle (infantry); 2 artillery shell (artillery); 3 tank (armor); 4 propeller plane (air force); 5 anchor (navy); 6 radio with a magnifying glass (intelligence); 7 shovel and dynamite (engineer); 8 red cross (medical); 9 star with two crossed flags (allied).",
"post": {
"kind": "grid",
"cols": 3,
"rows": 3,
"names": [
"tag_infantry",
"tag_artillery",
"tag_armor",
"tag_air",
"tag_navy",
"tag_intel",
"tag_engineer",
"tag_medical",
"tag_allied"
],
"folder": "ui",
"w": 32,
"h": 32
},
"no": 13
},
{
"name": "card_overlays",
"phase": "P1",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE image with SIX semi-transparent card overlay layers for a retro 2D card-and-survival game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nNo skulls, no blood. TRANSPARENCY: real transparent background (PNG alpha), never a painted checkerboard. Use pixel-art stepped opacity in hard-edged blocks, about 5 levels (0, 25, 50, 75, 100%): solid objects and blast cores 100%, glow halos 50%, smoke 25-75%, ground shadows and warning rings 25-50%. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nCANVAS: 1:1 square split into an invisible grid of 3 columns x 2 rows; each cell holds one 5:7 portrait overlay (laid over a card in the game), mostly 25-75% opacity so the card underneath stays faintly visible. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nREADING ORDER: 1 SMOKE: gray pixel smoke clouds with broken dithered edges, denser toward the bottom. 2 STATIC: radio interference, green and magenta pixel noise lines, scanlines, color-offset blocks. 3 SNOW: white pixel flakes of different sizes with a 25% blue tint. 4 NIGHT: deep blue-black 50-75% fill with a few star-like pixel specks. 5 CRACK: cracks spreading from the center with thin smoke wisps (failure result). 6 SUCCESS: golden pixel glow border with sparkles (success result).",
"post": {
"kind": "grid",
"cols": 3,
"rows": 2,
"names": [
"overlay_smoke",
"overlay_static",
"overlay_snow",
"overlay_night",
"overlay_crack",
"overlay_success"
],
"folder": "ui",
"w": 120,
"h": 168,
"alpha": "steps",
"fill": "stretch"
},
"no": 14
},
{
"name": "card_art_p",
"phase": "P1",
"size": "1024x1024",
"transparent": false,
"prompt": "Create ONE image with 16 card illustrations for a retro 2D card game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2.5x2.5 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nCANVAS: 1:1 square split into an invisible 4x4 grid of equal square cells. Each cell is one full-bleed square illustration (like a 104x104-pixel card picture enlarged): one centered subject with a strong readable silhouette on a simple scene background, bright heroic palette. No borders, frames or gaps. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nREADING ORDER (left to right, top to bottom):\nP01: rifle squad firing a volley in a line\nP02: three grenades in mid-air along an arc\nP03: mortar crew firing, shell trail rising\nP04: bomber seen from above dropping a bomb\nP05: warship firing its guns toward a coast\nP06: tank platoon advancing\nP07: combat engineer planting a time bomb with a red LED\nP08: field medical tent with a red cross\nP09: UN flags and soldiers of several nations standing together\nP10: scout with binoculars on a ridge\nP11: radio operator repairing a field radio with sparks\nP12: war-room map with pins and a glowing golden plan document\nP13: reconnaissance report papers with an aerial photo\nP14: officers gathered around a map table\nP15: supply crates descending by parachute\nP16: gold medal on a ribbon above a glowing star",
"post": {
"kind": "grid",
"cols": 4,
"rows": 4,
"names": [
"P01",
"P02",
"P03",
"P04",
"P05",
"P06",
"P07",
"P08",
"P09",
"P10",
"P11",
"P12",
"P13",
"P14",
"P15",
"P16"
],
"folder": "cards",
"w": 104,
"h": 104,
"alpha": "none",
"fill": "stretch"
},
"no": 15
},
{
"name": "card_art_m",
"phase": "P1",
"size": "1024x1024",
"transparent": false,
"prompt": "Create ONE image with 8 card illustrations for a retro 2D card game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2.5x2.5 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nCANVAS: 1:1 square split into an invisible 3x3 grid of equal square cells. Use the first 8 cells; the remaining cells stay completely empty. Each cell is one full-bleed square illustration (like a 104x104-pixel card picture enlarged): one centered subject with a strong readable silhouette on a simple scene background, darker, gloomier palette. No borders, frames or gaps. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nREADING ORDER (left to right, top to bottom):\nM01: false intelligence report with a big red X stamp\nM02: friendly fire: a shell landing near our own trench, warning triangle\nM03: empty supply crate and torn tarp\nM04: dud shell stuck in the ground with thin smoke\nM05: tired soldier sitting in the rain, lowered flag\nM06: frosted boots in deep snow with icicles\nM07: broken radio with a cut wire\nM08: mixed-up order papers flying in confusion",
"post": {
"kind": "grid",
"cols": 3,
"rows": 3,
"names": [
"M01",
"M02",
"M03",
"M04",
"M05",
"M06",
"M07",
"M08"
],
"folder": "cards",
"w": 104,
"h": 104,
"alpha": "none",
"fill": "stretch"
},
"no": 16
},
{
"name": "icons_ui",
"phase": "P1",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE image with 9 small UI icons for a retro 2D survival game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 8x8 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nNo skulls, no blood. BACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nCANVAS: 1:1 square split into an invisible grid of 3 columns x 3 rows; one icon per cell, centered, each designed as a bold 32x32-pixel icon enlarged, thick outline, simple shape readable at small size, consistent style across all icons. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nREADING ORDER (left to right, top to bottom): 1 small glowing supply star (experience pickup); 2 round red wax seal stamp with a star (guaranteed success); 3 small rifle bullet flying right; 4 artillery shell flying right; 5 bronze merit medal; 6 lightning bolt in a gear (tactical points); 7 folded operation map (operation experience); 8 padlock; 9 small blue-teal allied pennant on a short pole.",
"post": {
"kind": "grid",
"cols": 3,
"rows": 3,
"names": [
"star_xp",
"seal",
"bullet",
"shell",
"merit",
"tp",
"op",
"lock",
"flag_ally"
],
"folder": "ui",
"w": 32,
"h": 32
},
"no": 17
},
{
"name": "icons_upgrades",
"phase": "P1",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE image with 16 upgrade icons for a retro 2D survival game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 8x8 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nNo skulls, no blood. BACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nCANVAS: 1:1 square split into an invisible grid of 4 columns x 4 rows; one icon per cell, centered, each designed as a bold 32x32-pixel icon enlarged, thick outline, simple shape readable at small size, consistent style across all icons. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nREADING ORDER (left to right, top to bottom): 1 red heart with a plus; 2 steel helmet; 3 boot with wind lines; 4 rifle bullet with a power flash; 5 ammo belt with motion lines; 6 binoculars; 7 golden crosshair; 8 cracked target with a starburst; 9 horseshoe magnet pulling a star; 10 four-leaf clover on a dog tag; 11 three fanned cards with a plus; 12 card with a stopwatch; 13 M1 rifle; 14 hand grenade; 15 mortar tube; 16 two circular arrows.",
"post": {
"kind": "grid",
"cols": 4,
"rows": 4,
"names": [
"icon_upgrade_hp",
"icon_upgrade_defense",
"icon_upgrade_speed",
"icon_upgrade_attack",
"icon_upgrade_attack_speed",
"icon_upgrade_range",
"icon_upgrade_crit_chance",
"icon_upgrade_crit_damage",
"icon_upgrade_magnet",
"icon_upgrade_luck",
"icon_upgrade_hand_plus",
"icon_upgrade_draw_speed",
"icon_upgrade_weapon_rifle",
"icon_upgrade_weapon_grenade",
"icon_upgrade_weapon_mortar",
"icon_upgrade_reroll"
],
"folder": "ui",
"w": 32,
"h": 32
},
"no": 18
},
{
"name": "icons_skills",
"phase": "P1",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE image with 12 skill icons for a retro 2D survival game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 8x8 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nNo skulls, no blood. BACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nCANVAS: 1:1 square split into an invisible grid of 4 columns x 3 rows; one icon per cell, centered, each designed as a bold 32x32-pixel icon enlarged, thick outline, simple shape readable at small size, consistent style across all icons. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nREADING ORDER (left to right, top to bottom): 1 carpet bombing: three bombs falling in a row; 2 precision strike: crosshair over a single large bomb; 3 strafing run: fighter plane with a dotted bullet line; 4 naval salvo: warship gun turret firing; 5 artillery barrage: howitzer with three shells; 6 allied battalion: three helmets side by side with a small flag; 7 armor breakthrough: tank front view with speed lines; 8 airdrop: parachute with a supply crate; 9 total assault: starburst explosion with crossed rifles; 10 hold the line: steel shield dome; 11 encirclement: ring of inward arrows; 12 joint strike: plane, shell and rifle joined in a triangle.",
"post": {
"kind": "grid",
"cols": 4,
"rows": 3,
"names": [
"AS01",
"AS02",
"AS03",
"AS04",
"AS05",
"AS06",
"AS07",
"AS08",
"AS09",
"AS10",
"AS11",
"AS12"
],
"folder": "skills",
"w": 32,
"h": 32
},
"no": 19
},
{
"name": "vfx_1",
"phase": "P1",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE skill-effect animation sheet containing FIVE effects for a retro 2D survival game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark outline on solid objects only (not on smoke or glow); cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Bold, dramatic cartoon timing. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nNo blood or gore.\nTRANSPARENCY: real transparent background (PNG alpha), never a painted checkerboard. Use pixel-art stepped opacity in hard-edged blocks, about 5 levels (0, 25, 50, 75, 100%): solid objects and blast cores 100%, glow halos 50%, smoke 25-75%, ground shadows and warning rings 25-50%. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 5 rows of equal cells. Each row is one effect animated in 6 frames, left to right, with a clear start-to-end progression; rows from top to bottom in the order listed. Each effect is centered in its cell with the same ground-contact point in every frame of its row; small effects use about half of the cell, large ones fill most of it; nothing crosses a cell border. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nROWS:\nROW 1: V01 BOMB_FALL: 1-2 pulsing red dashed warning ring on the ground (50%); 3-5 finned bomb descending, scale 160% to 100%, shrinking ground shadow (50%), thin speed lines; 6 touchdown, white flash\nROW 2: V02 EXPLOSION_L: 1 white core flash; 2 orange fireball grows with a yellow core; 3 fireball peak with a pixel shockwave ring (50%); 4 fireball breaks into smoke and debris; 5 gray smoke plume rises (75%) with embers; 6 smoke thins (25%), scorched crater decal remains (50%)\nROW 3: V03 EXPLOSION_S: the same 6-step progression as the big explosion, but smaller and faster\nROW 4: V04 BOMBER: twin-engine propeller bomber seen from directly above, flying toward the bottom, propellers spinning, 6-frame loop\nROW 5: V04 BOMBER_SHADOW: the identical bomber silhouette as a flat black shape at 50% opacity, same 6-frame animation",
"post": {
"kind": "fx",
"rows": [
{
"name": "bomb_fall",
"frame": 100
},
{
"name": "explosion_l",
"frame": 200
},
{
"name": "explosion_s",
"frame": 100
},
{
"name": "bomber",
"frame": 200
},
{
"name": "bomber_shadow",
"frame": 200
}
]
},
"no": 20
},
{
"name": "vfx_2",
"phase": "P1",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE skill-effect animation sheet containing FIVE effects for a retro 2D survival game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark outline on solid objects only (not on smoke or glow); cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Bold, dramatic cartoon timing. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nNo blood or gore.\nTRANSPARENCY: real transparent background (PNG alpha), never a painted checkerboard. Use pixel-art stepped opacity in hard-edged blocks, about 5 levels (0, 25, 50, 75, 100%): solid objects and blast cores 100%, glow halos 50%, smoke 25-75%, ground shadows and warning rings 25-50%. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 5 rows of equal cells. Each row is one effect animated in 6 frames, left to right, with a clear start-to-end progression; rows from top to bottom in the order listed. Each effect is centered in its cell with the same ground-contact point in every frame of its row; small effects use about half of the cell, large ones fill most of it; nothing crosses a cell border. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nROWS:\nROW 1: V05 TIMEBOMB: 1 charge placed with a dust puff; 2 red LED slow blink; 3 medium blink; 4 fast blink with sparks; 5 white flash; 6 first blast frame\nROW 2: V06 BURST_SHELL: 1-2 colored smoke flare rising (50-75%); 3-6 small-to-medium shell impact with debris\nROW 3: V07 SUPPORT_VOLLEY: 1 allied silhouettes fade in (25-75%); 2 three UN riflemen in mixed uniforms with small flag patches crouch to aim; 3-5 firing with muzzle flashes and ejected casings; 6 fade out\nROW 4: V12 HIT_SPARK: 1 white flash star; 2-4 yellow sparks radiating; 5-6 fast fade\nROW 5: AS03 STRAFE (large, fills most of each cell): a fighter plane seen from above passes; a line of bullet-impact dust dots and muzzle glare",
"post": {
"kind": "fx",
"rows": [
{
"name": "timebomb",
"frame": 100
},
{
"name": "burst_shell",
"frame": 100
},
{
"name": "support_volley",
"frame": 200
},
{
"name": "hit_spark",
"frame": 100
},
{
"name": "skill_AS03",
"frame": 200
}
]
},
"no": 21
},
{
"name": "vfx_3",
"phase": "P1",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE skill-effect animation sheet containing FIVE effects for a retro 2D survival game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark outline on solid objects only (not on smoke or glow); cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Bold, dramatic cartoon timing. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nNo blood or gore.\nTRANSPARENCY: real transparent background (PNG alpha), never a painted checkerboard. Use pixel-art stepped opacity in hard-edged blocks, about 5 levels (0, 25, 50, 75, 100%): solid objects and blast cores 100%, glow halos 50%, smoke 25-75%, ground shadows and warning rings 25-50%. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 5 rows of equal cells. Each row is one effect animated in 6 frames, left to right, with a clear start-to-end progression; rows from top to bottom in the order listed. Each effect is centered in its cell with the same ground-contact point in every frame of its row; small effects use about half of the cell, large ones fill most of it; nothing crosses a cell border. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nROWS:\nROW 1: V08 NAVAL_SHELL: 1 big muzzle glow at the cell edge; 2-4 shell arcing with a thin smoke trail (50%); 5-6 impact with a tall dirt-and-smoke column\nROW 2: V09 TANK_CHARGE: tank rolling right with animated treads, lingering semi-transparent dust trail (25-50%), pixel shock arcs ahead, 6-frame loop\nROW 3: V10 SUPPLY_DROP: 1-3 parachute crate descending; 4 landing dust; 5-6 green-white healing sparkles rising (50-100%)\nROW 4: V11 MINES: 1-2 small mines drop and bury; 3 flash; 4-6 chain of small pops\nROW 5: V13 CRIT_RING: golden shock ring with radial lines expanding then fading over 6 frames",
"post": {
"kind": "fx",
"rows": [
{
"name": "naval_shell",
"frame": 200
},
{
"name": "tank_charge",
"frame": 200
},
{
"name": "supply_drop",
"frame": 100
},
{
"name": "mines",
"frame": 100
},
{
"name": "crit_ring",
"frame": 100
}
]
},
"no": 22
},
{
"name": "vfx_4",
"phase": "P1",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE skill-effect animation sheet containing FIVE effects for a retro 2D survival game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark outline on solid objects only (not on smoke or glow); cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Bold, dramatic cartoon timing. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nNo blood or gore.\nTRANSPARENCY: real transparent background (PNG alpha), never a painted checkerboard. Use pixel-art stepped opacity in hard-edged blocks, about 5 levels (0, 25, 50, 75, 100%): solid objects and blast cores 100%, glow halos 50%, smoke 25-75%, ground shadows and warning rings 25-50%. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 5 rows of equal cells. Each row is one effect animated in 6 frames, left to right, with a clear start-to-end progression; rows from top to bottom in the order listed. Each effect is centered in its cell with the same ground-contact point in every frame of its row; small effects use about half of the cell, large ones fill most of it; nothing crosses a cell border. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nROWS:\nROW 1: V14 MUZZLE_FLASH: small flash and a popping shell casing over 6 frames\nROW 2: V15 DUST_PUFF: soft dithered dust ring (25-50%) expanding and thinning over 6 frames\nROW 3: AS01 CARPET_BOMB (large, fills most of each cell): a bomber shadow sweeps across; three lines of explosions follow with smoke trails\nROW 4: AS04 NAVAL_SALVO (large, fills most of each cell): three wide muzzle flashes at the edge, then three shell impacts\nROW 5: AS06 ALLIED_BATTALION (large, fills most of each cell): a row of five UN riflemen with small flag patches fade in, fire, fade out",
"post": {
"kind": "fx",
"rows": [
{
"name": "muzzle",
"frame": 100
},
{
"name": "dust",
"frame": 100
},
{
"name": "skill_AS01",
"frame": 200
},
{
"name": "skill_AS04",
"frame": 200
},
{
"name": "skill_AS06",
"frame": 200
}
]
},
"no": 23
},
{
"name": "vfx_5",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE skill-effect animation sheet containing FIVE effects for a retro 2D survival game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark outline on solid objects only (not on smoke or glow); cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Bold, dramatic cartoon timing. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nNo blood or gore.\nTRANSPARENCY: real transparent background (PNG alpha), never a painted checkerboard. Use pixel-art stepped opacity in hard-edged blocks, about 5 levels (0, 25, 50, 75, 100%): solid objects and blast cores 100%, glow halos 50%, smoke 25-75%, ground shadows and warning rings 25-50%. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 5 rows of equal cells. Each row is one effect animated in 6 frames, left to right, with a clear start-to-end progression; rows from top to bottom in the order listed. Each effect is centered in its cell with the same ground-contact point in every frame of its row; small effects use about half of the cell, large ones fill most of it; nothing crosses a cell border. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nROWS:\nROW 1: AS07 ARMOR_CHARGE (large, fills most of each cell): three tanks abreast with dust trails and shock arcs\nROW 2: AS09 TOTAL_ASSAULT (large, fills most of each cell): cascading explosions inside a golden pixel ring (50%)\nROW 3: AS10 DEFENSE_DOME (large, fills most of each cell): translucent steel-blue dome (fill 25%, rim 75%) with scanline pixels and an impact ripple\nROW 4: AS11 ENCIRCLE (large, fills most of each cell): a contracting ring (75%) with inward arrows, then an implosion flash\nROW 5: AS12 JOINT_STRIKE (large, fills most of each cell): bomber shadow, shell trail and volley glow merge into one white flash",
"post": {
"kind": "fx",
"rows": [
{
"name": "skill_AS07",
"frame": 200
},
{
"name": "skill_AS09",
"frame": 200
},
{
"name": "skill_AS10",
"frame": 200
},
{
"name": "skill_AS11",
"frame": 200
},
{
"name": "skill_AS12",
"frame": 200
}
]
},
"no": 24
},
{
"name": "props_1",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE animation sheet of background props for a retro 2D survival game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nNo people, no blood or gore.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 5 rows of equal cells. Each row is one prop animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: each row shows the same prop in side view, centered in its cell, base or roots on one shared baseline 8% above the cell bottom, same scale in every frame, fully inside its cell, about 80% of the cell height or width. Only the animated parts move, by a few pixels. Frame 6 loops back to frame 1 smoothly.\nROWS:\nROW 1: broadleaf tree in summer green, leaves sway in the wind\nROW 2: tall pine tree, branches sway gently\nROW 3: burnt dead tree, bare branches tremble, a few ash flakes drift\nROW 4: snow-laden pine, snow dust falls from the branches\nROW 5: tall grass tuft waving in the wind",
"post": {
"kind": "props",
"rows": [
{
"name": "tree_leaf",
"frame": 200,
"fit": "h",
"px": 150
},
{
"name": "tree_pine",
"frame": 200,
"fit": "h",
"px": 160
},
{
"name": "tree_dead",
"frame": 200,
"fit": "h",
"px": 140
},
{
"name": "tree_snow",
"frame": 200,
"fit": "h",
"px": 160
},
{
"name": "grass",
"frame": 100,
"fit": "h",
"px": 30
}
]
},
"no": 25
},
{
"name": "map_korea",
"phase": "P2",
"size": "1024x1536",
"transparent": false,
"prompt": "Create ONE stage-select map of the Korean peninsula for a retro 2D game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 3x3 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; earthy sepia-olive wartime map palette for the whole sheet. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCANVAS: 1024x1536 portrait, top-down map. Accurate overall peninsula shape: jagged west coast with many small islands, smoother east coast, Taebaek mountains along the east, Jeju island in the south. Ocean in dark teal with simple pixel waves; tiny pixel mountains and rivers.\nDivide the land into 10 regions with thin dark borders and a distinct muted tint each, south to north: 1 Gyeongsang (southeast), 2 Jeolla (southwest), 3 Chungcheong, 4 Gangwon (east), 5 Gyeonggi, 6 Seoul (small city marker inside Gyeonggi), 7 Hwanghae, 8 Pyongan, 9 Pyongyang (small city marker inside Pyongan), 10 Hamgyong (northeast). Put one small empty round badge at the center of each region for in-game icons. A few decorative war-map touches (compass rose, torn paper edge), but absolutely no text, letters or numbers.",
"post": {
"kind": "map",
"w": 360,
"h": 540
},
"no": 26
},
{
"name": "tiles_2",
"phase": "P2",
"size": "1024x1024",
"transparent": false,
"prompt": "Create ONE image containing FOUR seamless ground textures for a retro 2D top-down survival game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 8x8 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCANVAS: 1:1 square split into a 2x2 grid of four equal square tiles, edge to edge, no gaps or borders. Each tile is a 64x64-pixel texture enlarged 8 times, viewed straight down from above (90 degrees), and wraps seamlessly on its own: its left edge continues its right edge and its top edge continues its bottom edge.\nLOOK: flat ground only, medium-dark values and LOW contrast so bright character sprites stand out on top; no outlines on the ground texture itself, no large objects, no strong shadows, no people, no blood; 4-6 small scattered details per tile (pebbles, tufts, cracks, puddles) spread evenly with no obvious repeat. Reading order: top-left, top-right, bottom-left, bottom-right.\nTILES:\n1. T05: rolling hill grass, short and dry (Chungcheong)\n2. T06: packed brown dirt country road with gravel\n3. T07: forest floor of fallen autumn leaves (Gangwon)\n4. T08: rocky valley stone path with moss",
"post": {
"kind": "tiles",
"ids": [
"T05",
"T06",
"T07",
"T08"
]
},
"no": 27
},
{
"name": "tiles_3",
"phase": "P2",
"size": "1024x1024",
"transparent": false,
"prompt": "Create ONE image containing FOUR seamless ground textures for a retro 2D top-down survival game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 8x8 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCANVAS: 1:1 square split into a 2x2 grid of four equal square tiles, edge to edge, no gaps or borders. Each tile is a 64x64-pixel texture enlarged 8 times, viewed straight down from above (90 degrees), and wraps seamlessly on its own: its left edge continues its right edge and its top edge continues its bottom edge.\nLOOK: flat ground only, medium-dark values and LOW contrast so bright character sprites stand out on top; no outlines on the ground texture itself, no large objects, no strong shadows, no people, no blood; 4-6 small scattered details per tile (pebbles, tufts, cracks, puddles) spread evenly with no obvious repeat. Reading order: top-left, top-right, bottom-left, bottom-right.\nTILES:\n1. T09: scrubby hillside grass with bare patches (Gyeonggi)\n2. T10: ruined village ground: dirt and broken roof-tile shards\n3. T11: cracked city asphalt with faded paint remnants (Seoul)\n4. T12: collapsed brick and concrete rubble",
"post": {
"kind": "tiles",
"ids": [
"T09",
"T10",
"T11",
"T12"
]
},
"no": 28
},
{
"name": "ally_A04",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A04 student volunteer soldier: young face, oversized helmet, dark school jacket under web gear, scarf, carbine. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A04",
"faction": "ally",
"tag": "S"
},
"no": 29
},
{
"name": "ally_A05",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A05 ROK Army grenadier: grenade bandolier across the chest, pouches, big overhand throwing arm. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A05",
"faction": "ally",
"tag": "S"
},
"no": 30
},
{
"name": "ally_A06",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A06 ROK military police: white helmet band, armband, whistle, baton and pistol. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A06",
"faction": "ally",
"tag": "S"
},
"no": 31
},
{
"name": "ally_A07",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A07 US Marine BAR gunner: green utility uniform, camouflage helmet cover, Browning Automatic Rifle. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A07",
"faction": "ally",
"tag": "S"
},
"no": 32
},
{
"name": "ally_A08",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A08 US Army bazooka trooper: long M20 bazooka tube on the shoulder, helmet, rocket bag. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A08",
"faction": "ally",
"tag": "S"
},
"no": 33
},
{
"name": "ally_A09",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A09 US Army medic: red-cross armband and helmet mark, medical bag, pistol holster. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A09",
"faction": "ally",
"tag": "S"
},
"no": 34
},
{
"name": "ally_A10",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A10 ROK Army machine gunner: light machine gun with bipod, ammo belt across the chest. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A10",
"faction": "ally",
"tag": "S"
},
"no": 35
},
{
"name": "ally_A11",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A11 ROK combat engineer: demolition pack, mine detector, small time bomb with a red LED. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A11",
"faction": "ally",
"tag": "S"
},
"no": 36
},
{
"name": "ally_A12",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A12 ROK Army sniper: scoped rifle, helmet netting with leaves, scarf. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A12",
"faction": "ally",
"tag": "S"
},
"no": 37
},
{
"name": "ally_A13",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A13 US Army Ranger: lean build, dark fatigues, Thompson submachine gun, combat knife. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A13",
"faction": "ally",
"tag": "S"
},
"no": 38
},
{
"name": "ally_A14",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A14 US artillery forward observer: radio backpack with tall antenna, binoculars, signal flare. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A14",
"faction": "ally",
"tag": "S"
},
"no": 39
},
{
"name": "ally_A15",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A15 US tank crewman: leather tanker helmet, goggles, coveralls, big wrench, pistol. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A15",
"faction": "ally",
"tag": "S"
},
"no": 40
},
{
"name": "enemy_E06",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E06 KPA occupation rifleman: khaki uniform, red armband, rifle. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E06",
"faction": "enemy",
"tag": "S"
},
"no": 41
},
{
"name": "enemy_E07",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E07 KPA sniper: scoped rifle, netting on the cap. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E07",
"faction": "enemy",
"tag": "S"
},
"no": 42
},
{
"name": "enemy_E08",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E08 KPA machine-gun team member: heavy machine gun with a gun shield on wheels. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E08",
"faction": "enemy",
"tag": "S"
},
"no": 43
},
{
"name": "enemy_E09",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E09 SU-76 self-propelled gun: open-top light armored gun. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: vehicle, aircraft or emplacement: about 85% of the cell width (aircraft fly level, their lowest point on the baseline). Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame loop: treads rolling or wheels turning with exhaust flicker; aircraft: a gentle flight wobble with engine glow\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 hatch closes or crew ducks; 3 armor plates flash steel-blue; 4 hold with a faint shimmer; 5 reopen\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E09",
"faction": "enemy",
"tag": "L"
},
"no": 44
},
{
"name": "enemy_E10",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E10 occupation division commander: long coat, peaked cap, medals, riding crop. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: boss: bulkier and taller, about 80% of the cell height, medals and long coat. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E10",
"faction": "enemy",
"tag": "B"
},
"no": 45
},
{
"name": "enemy_E11",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E11 KPA sapper: satchel charges, mines, shovel. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E11",
"faction": "enemy",
"tag": "S"
},
"no": 46
},
{
"name": "enemy_E12",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E12 KPA anti-tank gunner: long anti-tank rifle. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E12",
"faction": "enemy",
"tag": "S"
},
"no": 47
},
{
"name": "enemy_E13",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E13 courier motorcycle with sidecar: rider in goggles. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: vehicle, aircraft or emplacement: about 85% of the cell width (aircraft fly level, their lowest point on the baseline). Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame loop: treads rolling or wheels turning with exhaust flicker; aircraft: a gentle flight wobble with engine glow\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 hatch closes or crew ducks; 3 armor plates flash steel-blue; 4 hold with a faint shimmer; 5 reopen\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E13",
"faction": "enemy",
"tag": "L"
},
"no": 48
},
{
"name": "enemy_E14",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E14 T-34/85 tank: bigger turret, red star. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: vehicle, aircraft or emplacement: about 85% of the cell width (aircraft fly level, their lowest point on the baseline). Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame loop: treads rolling or wheels turning with exhaust flicker; aircraft: a gentle flight wobble with engine glow\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 hatch closes or crew ducks; 3 armor plates flash steel-blue; 4 hold with a faint shimmer; 5 reopen\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E14",
"faction": "enemy",
"tag": "L"
},
"no": 49
},
{
"name": "enemy_E15",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E15 artillery regiment commander: field telephone, shell crate, peaked cap. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: boss: bulkier and taller, about 80% of the cell height, medals and long coat. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E15",
"faction": "enemy",
"tag": "B"
},
"no": 50
},
{
"name": "enemy_E16",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E16 mountain infiltrator: leaves on the helmet, submachine gun, knife. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E16",
"faction": "enemy",
"tag": "S"
},
"no": 51
},
{
"name": "enemy_E17",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E17 KPA grenadier: grenade belts, overhand throw. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E17",
"faction": "enemy",
"tag": "S"
},
"no": 52
},
{
"name": "enemy_E18",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E18 flamethrower trooper: flamethrower backpack, orange pixel flame jet. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E18",
"faction": "enemy",
"tag": "S"
},
"no": 53
},
{
"name": "enemy_E19",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E19 76mm mountain gun emplacement: gun behind sandbags, one crew silhouette. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: vehicle, aircraft or emplacement: about 85% of the cell width (aircraft fly level, their lowest point on the baseline). Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame loop: treads rolling or wheels turning with exhaust flicker; aircraft: a gentle flight wobble with engine glow\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 hatch closes or crew ducks; 3 armor plates flash steel-blue; 4 hold with a faint shimmer; 5 reopen\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E19",
"faction": "enemy",
"tag": "L"
},
"no": 54
},
{
"name": "enemy_E20",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E20 mountain defense commander: fur-collar coat, binoculars. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: boss: bulkier and taller, about 80% of the cell height, medals and long coat. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E20",
"faction": "enemy",
"tag": "B"
},
"no": 55
},
{
"name": "enemy_E21",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E21 KPA assault trooper: drum-magazine submachine gun, sprinting. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E21",
"faction": "enemy",
"tag": "S"
},
"no": 56
},
{
"name": "enemy_E22",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E22 KPA mortar squad member: large mortar with an ammo carrier. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E22",
"faction": "enemy",
"tag": "S"
},
"no": 57
},
{
"name": "enemy_E23",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E23 KPA anti-aircraft gunner: AA machine gun on a tripod aimed upward. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E23",
"faction": "enemy",
"tag": "S"
},
"no": 58
},
{
"name": "enemy_E24",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E24 heavy self-propelled gun: large armored gun carriage. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: vehicle, aircraft or emplacement: about 85% of the cell width (aircraft fly level, their lowest point on the baseline). Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame loop: treads rolling or wheels turning with exhaust flicker; aircraft: a gentle flight wobble with engine glow\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 hatch closes or crew ducks; 3 armor plates flash steel-blue; 4 hold with a faint shimmer; 5 reopen\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E24",
"faction": "enemy",
"tag": "L"
},
"no": 59
},
{
"name": "enemy_E25",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E25 corps commander: sash, medals, peaked cap, swagger stick. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: boss: bulkier and taller, about 80% of the cell height, medals and long coat. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E25",
"faction": "enemy",
"tag": "B"
},
"no": 60
},
{
"name": "npc_N03",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of a friendly NPC for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Kind, warm expressions; no injury or distress. A speech bubble may contain three dots only.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: N03 supply truck soldier: work cap, carrying an ammo crate, small hand cart.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Use only the top 3 rows (each row is one action animated in 5 frames, left to right); rows 4 and 5 stay completely empty. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 15 used cells, as if traced from one model sheet. Side view facing RIGHT toward the player. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 idle (5 frames): gentle breathing, small weight shift, a blink on frame 4, a tiny prop movement (cloth, strap, tool)\nROW 2 talk (5 frames): nod and head tilt, one hand gesturing, mouth opening and closing, a small three-dot speech bubble on frames 2-4\nROW 3 give (5 frames): 1 notice the player; 2 reach into bag or pocket; 3 extend an arm holding a small item with a white-gold glint; 4 item sparkles; 5 pull back with a small friendly wave",
"post": {
"kind": "unit",
"id": "N03",
"faction": "npc",
"tag": "S"
},
"no": 61
},
{
"name": "npc_N04",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of a friendly NPC for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Kind, warm expressions; no injury or distress. A speech bubble may contain three dots only.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: N04 elderly Korean villager: white hanbok-style clothes, straw hat, cane, cloth bundle; calm and dignified.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Use only the top 3 rows (each row is one action animated in 5 frames, left to right); rows 4 and 5 stay completely empty. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 15 used cells, as if traced from one model sheet. Side view facing RIGHT toward the player. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 idle (5 frames): gentle breathing, small weight shift, a blink on frame 4, a tiny prop movement (cloth, strap, tool)\nROW 2 talk (5 frames): nod and head tilt, one hand gesturing, mouth opening and closing, a small three-dot speech bubble on frames 2-4\nROW 3 give (5 frames): 1 notice the player; 2 reach into bag or pocket; 3 extend an arm holding a small item with a white-gold glint; 4 item sparkles; 5 pull back with a small friendly wave",
"post": {
"kind": "unit",
"id": "N04",
"faction": "npc",
"tag": "S"
},
"no": 62
},
{
"name": "npc_N05",
"phase": "P2",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of a friendly NPC for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Kind, warm expressions; no injury or distress. A speech bubble may contain three dots only.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: N05 young radio operator: oversized helmet, field radio with antenna, headset.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Use only the top 3 rows (each row is one action animated in 5 frames, left to right); rows 4 and 5 stay completely empty. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 15 used cells, as if traced from one model sheet. Side view facing RIGHT toward the player. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 idle (5 frames): gentle breathing, small weight shift, a blink on frame 4, a tiny prop movement (cloth, strap, tool)\nROW 2 talk (5 frames): nod and head tilt, one hand gesturing, mouth opening and closing, a small three-dot speech bubble on frames 2-4\nROW 3 give (5 frames): 1 notice the player; 2 reach into bag or pocket; 3 extend an arm holding a small item with a white-gold glint; 4 item sparkles; 5 pull back with a small friendly wave",
"post": {
"kind": "unit",
"id": "N05",
"faction": "npc",
"tag": "S"
},
"no": 63
},
{
"name": "tiles_4",
"phase": "P3",
"size": "1024x1024",
"transparent": false,
"prompt": "Create ONE image containing FOUR seamless ground textures for a retro 2D top-down survival game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 8x8 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCANVAS: 1:1 square split into a 2x2 grid of four equal square tiles, edge to edge, no gaps or borders. Each tile is a 64x64-pixel texture enlarged 8 times, viewed straight down from above (90 degrees), and wraps seamlessly on its own: its left edge continues its right edge and its top edge continues its bottom edge.\nLOOK: flat ground only, medium-dark values and LOW contrast so bright character sprites stand out on top; no outlines on the ground texture itself, no large objects, no strong shadows, no people, no blood; 4-6 small scattered details per tile (pebbles, tufts, cracks, puddles) spread evenly with no obvious repeat. Reading order: top-left, top-right, bottom-left, bottom-right.\nTILES:\n1. T13: reed plain: dry brown reeds and mud (Hwanghae)\n2. T14: dry yellow-brown loess plain with cracks\n3. T15: silver-grass plain and railway gravel (Pyongan)\n4. T16: muddy road after rain with puddles and tire tracks",
"post": {
"kind": "tiles",
"ids": [
"T13",
"T14",
"T15",
"T16"
]
},
"no": 64
},
{
"name": "tiles_5",
"phase": "P3",
"size": "1024x1024",
"transparent": false,
"prompt": "Create ONE image containing FOUR seamless ground textures for a retro 2D top-down survival game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 8x8 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCANVAS: 1:1 square split into a 2x2 grid of four equal square tiles, edge to edge, no gaps or borders. Each tile is a 64x64-pixel texture enlarged 8 times, viewed straight down from above (90 degrees), and wraps seamlessly on its own: its left edge continues its right edge and its top edge continues its bottom edge.\nLOOK: flat ground only, medium-dark values and LOW contrast so bright character sprites stand out on top; no outlines on the ground texture itself, no large objects, no strong shadows, no people, no blood; 4-6 small scattered details per tile (pebbles, tufts, cracks, puddles) spread evenly with no obvious repeat. Reading order: top-left, top-right, bottom-left, bottom-right.\nTILES:\n1. T17: paved city street with patches of old snow (Pyongyang)\n2. T18: river sandbar with wet sand ripples\n3. T19: snow-covered ground with faint footprints (Hamgyong)\n4. T20: frozen lake surface with ice cracks and snow dust",
"post": {
"kind": "tiles",
"ids": [
"T17",
"T18",
"T19",
"T20"
]
},
"no": 65
},
{
"name": "props_2",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE animation sheet of background props for a retro 2D survival game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nNo people, no blood or gore.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 5 rows of equal cells. Each row is one prop animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: each row shows the same prop in side view, centered in its cell, base or roots on one shared baseline 8% above the cell bottom, same scale in every frame, fully inside its cell, about 80% of the cell height or width. Only the animated parts move, by a few pixels. Frame 6 loops back to frame 1 smoothly.\nROWS:\nROW 1: silver-grass or reed clump rippling\nROW 2: low shrub, leaves flutter\nROW 3: tattered plain military flag on a pole fluttering, no emblem\nROW 4: steel oil drum with flickering fire on top\nROW 5: burnt-out truck wreck with a thin rising semi-transparent smoke column",
"post": {
"kind": "props",
"rows": [
{
"name": "reed",
"frame": 100,
"fit": "h",
"px": 40
},
{
"name": "bush",
"frame": 100,
"fit": "h",
"px": 34
},
{
"name": "flag",
"frame": 100,
"fit": "h",
"px": 80
},
{
"name": "drum_fire",
"frame": 100,
"fit": "h",
"px": 44
},
{
"name": "burnt_truck",
"frame": 200,
"fit": "w",
"px": 150
}
]
},
"no": 66
},
{
"name": "props_3",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE animation sheet of background props for a retro 2D survival game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nNo people, no blood or gore.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 5 rows of equal cells. Each row is one prop animated in 6 frames, left to right; rows from top to bottom in the order listed. Rows 4-5 stay completely empty. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: each row shows the same prop in side view, centered in its cell, base or roots on one shared baseline 8% above the cell bottom, same scale in every frame, fully inside its cell, about 80% of the cell height or width. Only the animated parts move, by a few pixels. Frame 6 loops back to frame 1 smoothly.\nROWS:\nROW 1: barbed-wire fence segment, slight wire shimmer\nROW 2: sandbag wall, only a tiny dust puff changes\nROW 3: mossy boulder, nearly static",
"post": {
"kind": "props",
"rows": [
{
"name": "wire",
"frame": 100,
"fit": "w",
"px": 90
},
{
"name": "sandbag",
"frame": 100,
"fit": "w",
"px": 80
},
{
"name": "rock",
"frame": 100,
"fit": "h",
"px": 36
}
]
},
"no": 67
},
{
"name": "ally_A16",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A16 British infantryman: Brodie helmet, khaki battledress, Lee-Enfield rifle. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A16",
"faction": "ally",
"tag": "S"
},
"no": 68
},
{
"name": "ally_A17",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A17 Turkish brigade assault trooper: small crescent-and-star patch, fixed bayonet, lunging stance. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A17",
"faction": "ally",
"tag": "S"
},
"no": 69
},
{
"name": "ally_A18",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A18 Canadian sniper: small maple-leaf patch, scoped rifle, camouflage scarf. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A18",
"faction": "ally",
"tag": "S"
},
"no": 70
},
{
"name": "ally_A19",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A19 Australian scout: slouch hat with one side of the brim pinned up, jungle-green uniform, light rifle. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A19",
"faction": "ally",
"tag": "S"
},
"no": 71
},
{
"name": "ally_A20",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A20 Ethiopian Kagnew Battalion soldier: khaki uniform, helmet, rifle, small green-yellow-red patch. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A20",
"faction": "ally",
"tag": "S"
},
"no": 72
},
{
"name": "ally_A21",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A21 Greek mountain trooper: beret, mountain pack with coiled rope, slung rifle, sturdy boots. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A21",
"faction": "ally",
"tag": "S"
},
"no": 73
},
{
"name": "ally_A22",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A22 US forward air controller: radio set, bright orange signal panel, helmet. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A22",
"faction": "ally",
"tag": "S"
},
"no": 74
},
{
"name": "ally_A23",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A23 US Navy gunfire spotter: navy dungarees, binoculars, field radio. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A23",
"faction": "ally",
"tag": "S"
},
"no": 75
},
{
"name": "ally_A24",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A24 Filipino infantryman: small Philippine flag patch, helmet, carbine, agile stance. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A24",
"faction": "ally",
"tag": "S"
},
"no": 76
},
{
"name": "ally_A25",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A25 French grenadier: helmet with a small tricolor patch, grenade bag, rifle grenade launcher. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A25",
"faction": "ally",
"tag": "S"
},
"no": 77
},
{
"name": "ally_A26",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A26 Dutch infantryman: small orange-white-blue patch, helmet, rifle, steady stance. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A26",
"faction": "ally",
"tag": "S"
},
"no": 78
},
{
"name": "ally_A27",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A27 Thai infantryman: small Thai flag patch, helmet, rifle, nimble dodge pose. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A27",
"faction": "ally",
"tag": "S"
},
"no": 79
},
{
"name": "ally_A28",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A28 ROK raider: dark fatigues, headband, submachine gun, knife. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A28",
"faction": "ally",
"tag": "S"
},
"no": 80
},
{
"name": "ally_A29",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A29 US helicopter pilot: flight helmet, goggles, flight suit, sidearm, signaling with a flare. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A29",
"faction": "ally",
"tag": "S"
},
"no": 81
},
{
"name": "ally_A30",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: A30 Allied staff officer (fictional, not a real person): officer cap, long coat, map case, binoculars, pointer baton. Ally unit (ROK Army, US forces or UN contingent); muted olive and khaki uniform colors with steel-gray metal.\nCANVAS: 1:1 square split into an invisible grid of 6 columns x 6 rows of equal square cells. Each row is one action animated in 6 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 36 cells, as if traced from one model sheet: identical design, colors, gear and size. Side view facing RIGHT. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Leave the empty space around the body for muzzle flashes and effects; nothing crosses a cell border. Each row loops smoothly from frame 6 back to frame 1.\nROWS:\nROW 1 attack (6 frames): 1 ready stance; 2 aim begins; 3 weapon raised, peak anticipation; 4 fire or strike with a bright yellow-white flash; 5 exaggerated recoil and follow-through; 6 settle back toward ready\nROW 2 defend (6 frames): 1 ready; 2 flinch and raise guard (arm, weapon, plate or helmet); 3 brace low; 4 block impact with a small white spark; 5 pushed back slightly, holding guard; 6 lower guard\nROW 3 skill (6 frames): 1 ready; 2 crouch and gather; 3 raise an arm, radio or signal toward the sky, small white-gold glint starts; 4 glint peak, point or throw forward; 5 glint fades; 6 recover\nROW 4 move_slow (6 frames): relaxed 6-frame walk cycle, weapon carried low, gentle 1-pixel bob\nROW 5 move_fast (6 frames): 6-frame run cycle, leaning forward, long strides, small dust puffs at the feet on frames 2 and 5\nROW 6 crit (6 frames): 1 ready; 2 deep wind-up with a thin golden outline; 3 freeze anticipation; 4 powerful strike with a big yellow-white flash and a pixel shock ring; 5 hold and recoil; 6 recover",
"post": {
"kind": "unit",
"id": "A30",
"faction": "ally",
"tag": "S"
},
"no": 82
},
{
"name": "enemy_E26",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E26 urban sniper: rubble-colored camouflage cloak, scoped rifle. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E26",
"faction": "enemy",
"tag": "S"
},
"no": 83
},
{
"name": "enemy_E27",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E27 barricade engineer: barbed-wire roll and sandbags. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E27",
"faction": "enemy",
"tag": "S"
},
"no": 84
},
{
"name": "enemy_E28",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E28 political officer: red armband, megaphone, faint red aura. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E28",
"faction": "enemy",
"tag": "S"
},
"no": 85
},
{
"name": "enemy_E29",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E29 heavy tank: thick armor, big gun, red star. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: vehicle, aircraft or emplacement: about 85% of the cell width (aircraft fly level, their lowest point on the baseline). Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame loop: treads rolling or wheels turning with exhaust flicker; aircraft: a gentle flight wobble with engine glow\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 hatch closes or crew ducks; 3 armor plates flash steel-blue; 4 hold with a faint shimmer; 5 reopen\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E29",
"faction": "enemy",
"tag": "L"
},
"no": 86
},
{
"name": "enemy_E30",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E30 capital defense commander: grand greatcoat, medals, binoculars. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: boss: bulkier and taller, about 80% of the cell height, medals and long coat. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E30",
"faction": "enemy",
"tag": "B"
},
"no": 87
},
{
"name": "enemy_E31",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E31 PVA infantryman: quilted khaki-green winter uniform, cloth cap with a red star, rifle, running. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E31",
"faction": "enemy",
"tag": "S"
},
"no": 88
},
{
"name": "enemy_E32",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E32 PVA scout: white cloak over khaki, low crouched run. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E32",
"faction": "enemy",
"tag": "S"
},
"no": 89
},
{
"name": "enemy_E33",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E33 PVA signal trooper: bugle and whistle, cloth cap. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E33",
"faction": "enemy",
"tag": "S"
},
"no": 90
},
{
"name": "enemy_E34",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E34 PVA mortar crewman: mortar and ammo boxes. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E34",
"faction": "enemy",
"tag": "S"
},
"no": 91
},
{
"name": "enemy_E35",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E35 PVA division commander: quilted greatcoat, medals, field glasses. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: boss: bulkier and taller, about 80% of the cell height, medals and long coat. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E35",
"faction": "enemy",
"tag": "B"
},
"no": 92
},
{
"name": "enemy_E36",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E36 PVA sniper: snow-camouflage cloak, scoped rifle. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E36",
"faction": "enemy",
"tag": "S"
},
"no": 93
},
{
"name": "enemy_E37",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E37 PVA anti-tank assault soldier: pole charge and satchel charge. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E37",
"faction": "enemy",
"tag": "S"
},
"no": 94
},
{
"name": "enemy_E38",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E38 MiG-15 jet fighter seen from the side: silver swept-wing jet, red star. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: vehicle, aircraft or emplacement: about 85% of the cell width (aircraft fly level, their lowest point on the baseline). Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame loop: treads rolling or wheels turning with exhaust flicker; aircraft: a gentle flight wobble with engine glow\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 hatch closes or crew ducks; 3 armor plates flash steel-blue; 4 hold with a faint shimmer; 5 reopen\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E38",
"faction": "enemy",
"tag": "L"
},
"no": 95
},
{
"name": "enemy_E39",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E39 truck-mounted rocket launcher: multi-rail launcher. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: vehicle, aircraft or emplacement: about 85% of the cell width (aircraft fly level, their lowest point on the baseline). Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame loop: treads rolling or wheels turning with exhaust flicker; aircraft: a gentle flight wobble with engine glow\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 hatch closes or crew ducks; 3 armor plates flash steel-blue; 4 hold with a faint shimmer; 5 reopen\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E39",
"faction": "enemy",
"tag": "L"
},
"no": 96
},
{
"name": "enemy_E40",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E40 PVA corps commander: winter greatcoat, fur hat, medals. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: boss: bulkier and taller, about 80% of the cell height, medals and long coat. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E40",
"faction": "enemy",
"tag": "B"
},
"no": 97
},
{
"name": "enemy_E41",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E41 guard infantryman: polished uniform, red collar tabs, submachine gun. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E41",
"faction": "enemy",
"tag": "S"
},
"no": 98
},
{
"name": "enemy_E42",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E42 special forces soldier: dark uniform, beret, knife and submachine gun. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E42",
"faction": "enemy",
"tag": "S"
},
"no": 99
},
{
"name": "enemy_E43",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E43 twin-barrel anti-aircraft gun emplacement: barrels pointing up. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: vehicle, aircraft or emplacement: about 85% of the cell width (aircraft fly level, their lowest point on the baseline). Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame loop: treads rolling or wheels turning with exhaust flicker; aircraft: a gentle flight wobble with engine glow\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 hatch closes or crew ducks; 3 armor plates flash steel-blue; 4 hold with a faint shimmer; 5 reopen\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E43",
"faction": "enemy",
"tag": "L"
},
"no": 100
},
{
"name": "enemy_E44",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E44 heavy tank with a small red pennant. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: vehicle, aircraft or emplacement: about 85% of the cell width (aircraft fly level, their lowest point on the baseline). Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame loop: treads rolling or wheels turning with exhaust flicker; aircraft: a gentle flight wobble with engine glow\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 hatch closes or crew ducks; 3 armor plates flash steel-blue; 4 hold with a faint shimmer; 5 reopen\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E44",
"faction": "enemy",
"tag": "L"
},
"no": 101
},
{
"name": "enemy_E45",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E45 supreme defense commander: ornate uniform, many medals, cane. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: boss: bulkier and taller, about 80% of the cell height, medals and long coat. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E45",
"faction": "enemy",
"tag": "B"
},
"no": 102
},
{
"name": "enemy_E46",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E46 winter elite infantryman: fur hat, white winter camouflage, sliding on ice. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E46",
"faction": "enemy",
"tag": "S"
},
"no": 103
},
{
"name": "enemy_E47",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E47 PVA mountain elite: snow-camouflage cloak, automatic rifle. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: infantry: the figure is about 60% of the cell height. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E47",
"faction": "enemy",
"tag": "S"
},
"no": 104
},
{
"name": "enemy_E48",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E48 pair of MiG jet fighters flying in formation, seen from the side. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: vehicle, aircraft or emplacement: about 85% of the cell width (aircraft fly level, their lowest point on the baseline). Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame loop: treads rolling or wheels turning with exhaust flicker; aircraft: a gentle flight wobble with engine glow\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 hatch closes or crew ducks; 3 armor plates flash steel-blue; 4 hold with a faint shimmer; 5 reopen\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E48",
"faction": "enemy",
"tag": "L"
},
"no": 105
},
{
"name": "enemy_E49",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E49 armored rocket vehicle: tank hull with a rocket pod. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: vehicle, aircraft or emplacement: about 85% of the cell width (aircraft fly level, their lowest point on the baseline). Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame loop: treads rolling or wheels turning with exhaust flicker; aircraft: a gentle flight wobble with engine glow\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 hatch closes or crew ducks; 3 armor plates flash steel-blue; 4 hold with a faint shimmer; 5 reopen\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E49",
"faction": "enemy",
"tag": "L"
},
"no": 106
},
{
"name": "enemy_E50",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of an ENEMY unit for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Bosses are generic fictional commanders identified only by rank and costume.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: E50 final boss, fictional northern coalition supreme commander: imposing heavy greatcoat and cape, many medals, glowing dark-red aura. Enemy unit (Korean People's Army or Chinese People's Volunteer Army) shown as a game opponent; khaki, brown or quilted green uniforms with red-star badges and dark-red accents so it reads as hostile.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Each row is one action animated in 5 frames, left to right; rows from top to bottom in the order listed. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME unit in all 25 cells, as if traced from one model sheet: identical design, colors and size. Side view facing RIGHT. Size: boss: bulkier and taller, about 80% of the cell height, medals and long coat. Horizontally centered, feet, wheels or treads on one shared baseline 8% above the cell bottom. Nothing crosses a cell border. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 move (5 frames): 5-frame march or run cycle; fast units lean forward with small dust puffs\nROW 2 attack (5 frames): 1 ready; 2 wind-up; 3 aim or raise weapon; 4 fire or strike with a flash; 5 recoil and recover\nROW 3 defend (5 frames): 1 ready; 2 flinch; 3 raise guard or crouch behind cover; 4 hold guard with a faint steel-blue shimmer; 5 lower guard\nROW 4 crit (5 frames): heavy attack: 1-2 charge-up with a growing red pixel warning glow; 3 powerful strike or heavy shot with a large flash; 4 follow-through; 5 recovery\nROW 5 hit (5 frames): damage reaction: 1 normal; 2 full white silhouette flash; 3 maximum recoil with squash; 4 wobble and stagger; 5 recover",
"post": {
"kind": "unit",
"id": "E50",
"faction": "enemy",
"tag": "B"
},
"no": 107
},
{
"name": "npc_N06",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of a friendly NPC for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Kind, warm expressions; no injury or distress. A speech bubble may contain three dots only.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: N06 interpreter: civilian jacket with an armband, round glasses, dictionary.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Use only the top 3 rows (each row is one action animated in 5 frames, left to right); rows 4 and 5 stay completely empty. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 15 used cells, as if traced from one model sheet. Side view facing RIGHT toward the player. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 idle (5 frames): gentle breathing, small weight shift, a blink on frame 4, a tiny prop movement (cloth, strap, tool)\nROW 2 talk (5 frames): nod and head tilt, one hand gesturing, mouth opening and closing, a small three-dot speech bubble on frames 2-4\nROW 3 give (5 frames): 1 notice the player; 2 reach into bag or pocket; 3 extend an arm holding a small item with a white-gold glint; 4 item sparkles; 5 pull back with a small friendly wave",
"post": {
"kind": "unit",
"id": "N06",
"faction": "npc",
"tag": "S"
},
"no": 108
},
{
"name": "npc_N07",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of a friendly NPC for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Kind, warm expressions; no injury or distress. A speech bubble may contain three dots only.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: N07 military chaplain: officer uniform, small plain cross badge, small book.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Use only the top 3 rows (each row is one action animated in 5 frames, left to right); rows 4 and 5 stay completely empty. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 15 used cells, as if traced from one model sheet. Side view facing RIGHT toward the player. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 idle (5 frames): gentle breathing, small weight shift, a blink on frame 4, a tiny prop movement (cloth, strap, tool)\nROW 2 talk (5 frames): nod and head tilt, one hand gesturing, mouth opening and closing, a small three-dot speech bubble on frames 2-4\nROW 3 give (5 frames): 1 notice the player; 2 reach into bag or pocket; 3 extend an arm holding a small item with a white-gold glint; 4 item sparkles; 5 pull back with a small friendly wave",
"post": {
"kind": "unit",
"id": "N07",
"faction": "npc",
"tag": "S"
},
"no": 109
},
{
"name": "npc_N08",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of a friendly NPC for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Kind, warm expressions; no injury or distress. A speech bubble may contain three dots only.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: N08 mechanic: coveralls, wrench, toolbox, goggles pushed up.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Use only the top 3 rows (each row is one action animated in 5 frames, left to right); rows 4 and 5 stay completely empty. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 15 used cells, as if traced from one model sheet. Side view facing RIGHT toward the player. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 idle (5 frames): gentle breathing, small weight shift, a blink on frame 4, a tiny prop movement (cloth, strap, tool)\nROW 2 talk (5 frames): nod and head tilt, one hand gesturing, mouth opening and closing, a small three-dot speech bubble on frames 2-4\nROW 3 give (5 frames): 1 notice the player; 2 reach into bag or pocket; 3 extend an arm holding a small item with a white-gold glint; 4 item sparkles; 5 pull back with a small friendly wave",
"post": {
"kind": "unit",
"id": "N08",
"faction": "npc",
"tag": "S"
},
"no": 110
},
{
"name": "npc_N09",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of a friendly NPC for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Kind, warm expressions; no injury or distress. A speech bubble may contain three dots only.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: N09 military police traffic controller: white helmet band, two signal batons.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Use only the top 3 rows (each row is one action animated in 5 frames, left to right); rows 4 and 5 stay completely empty. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 15 used cells, as if traced from one model sheet. Side view facing RIGHT toward the player. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 idle (5 frames): gentle breathing, small weight shift, a blink on frame 4, a tiny prop movement (cloth, strap, tool)\nROW 2 talk (5 frames): nod and head tilt, one hand gesturing, mouth opening and closing, a small three-dot speech bubble on frames 2-4\nROW 3 give (5 frames): 1 notice the player; 2 reach into bag or pocket; 3 extend an arm holding a small item with a white-gold glint; 4 item sparkles; 5 pull back with a small friendly wave",
"post": {
"kind": "unit",
"id": "N09",
"faction": "npc",
"tag": "S"
},
"no": 111
},
{
"name": "npc_N10",
"phase": "P3",
"size": "1024x1024",
"transparent": true,
"prompt": "Create ONE sprite animation sheet of a friendly NPC for a retro 2D survival action game set in the 1950 Korean War.\nART STYLE: hand-pixeled 16-bit arcade sprite art in the spirit of late-1990s 2D run-and-gun games. It must look like a LOW-RESOLUTION sprite enlarged with nearest-neighbor: every art pixel is a crisp, uniform square block (about 2x2 image pixels), aligned to one pixel grid across the whole image. 1-pixel dark-brown outline (#2A1A10) around every shape; cel shading with 3-4 flat tones per material; light from the upper left; small bright highlights; limited palette of about 32 colors for the whole sheet. Chunky heroic proportions (about 3 heads tall), big hands and boots, readable silhouettes, exaggerated hand-animated poses with squash and stretch. Forbidden: anti-aliasing, soft gradients, blur, glow bloom, painterly texture, 3D render, photo, text, letters, numbers, logos, watermark, signature.\nCONTENT RULES: no blood, gore, wounds or corpses (hits are white flashes, sparks and dust); no civilian harm; no real people; factions are shown only by uniform, gear and small flag patches, never by caricatured faces or ethnicity; respectful, heroic arcade tone. Kind, warm expressions; no injury or distress. A speech bubble may contain three dots only.\nBACKGROUND: real transparent background (PNG alpha), never a painted checkerboard, no floor, no cast ground shadow. If true transparency is impossible, use one flat pure magenta #FF00FF background instead.\nSUBJECT: N10 dog handler with a military working dog: handler on the right, shepherd-type dog on the left, leash.\nCANVAS: 1:1 square split into an invisible grid of 5 columns x 5 rows of equal square cells. Use only the top 3 rows (each row is one action animated in 5 frames, left to right); rows 4 and 5 stay completely empty. The grid is invisible: no grid lines, borders, numbers or labels anywhere.\nFRAMES: the SAME character in all 15 used cells, as if traced from one model sheet. Side view facing RIGHT toward the player. The figure is about 60% of the cell height, horizontally centered, feet on one shared baseline 8% above the cell bottom. Each row loops smoothly from frame 5 back to frame 1.\nROWS:\nROW 1 idle (5 frames): gentle breathing, small weight shift, a blink on frame 4, a tiny prop movement (cloth, strap, tool)\nROW 2 talk (5 frames): nod and head tilt, one hand gesturing, mouth opening and closing, a small three-dot speech bubble on frames 2-4\nROW 3 give (5 frames): 1 notice the player; 2 reach into bag or pocket; 3 extend an arm holding a small item with a white-gold glint; 4 item sparkles; 5 pull back with a small friendly wave",
"post": {
"kind": "unit",
"id": "N10",
"faction": "npc",
"tag": "S"
},
"no": 112
}
]''')
print(len(PROMPTS), "개 항목:", {p: sum(1 for it in PROMPTS if it["phase"] == p) for p in ("P1", "P2", "P3")})


In [ ]:
#@title 4) 후처리 코드 (건드리지 마세요) { display-mode: "form" }
# tools/art/postprocess.py 와 동일
"""AI 생성 원본 → 게임 규격 PNG 변환 (ASSETS.md 규격).

    python3 tools/art/postprocess.py --raw <원본 폴더> --out <출력 폴더> [--only ally_A01,tiles_1]

원본 파일명은 <항목 이름>.png (예: raw/ally_A01.png). 출력은 <out>/public/assets/... 구조라서
그대로 저장소 루트에 덮어 복사하면 게임이 자동으로 사용한다.

하는 일
  1. 투명 배경이 없으면 테두리의 지배적인 색(마젠타·단색·가짜 체크무늬)을 투명 처리
  2. 격자 칸 자르기
  3. 유닛: 기준 동작(걷기/이동/대기)의 몸체 크기로 배율을 정해 모든 프레임에 같은 변환 적용
     → 몸체 높이(보병 62px)·발 기준점(50, 92)·좌우 중심이 시트마다 일정
  4. 축소는 premultiplied BOX 필터, 알파는 경계 하드닝(이펙트는 5단계), 시트별 팔레트 제한
  5. 출력 시트는 4x4 격자(앞 N칸 사용), 품질 경고(report.json)와 미리보기 이미지 생성

Colab 노트북(generate_images.ipynb)에 이 파일 내용이 그대로 들어간다. 필요: Pillow, numpy.
"""


import json
import math
import os
import sys

import numpy as np
from PIL import Image

OUT_GRID = 4  # 출력 시트: 4x4 격자, 좌→우·위→아래로 앞 N칸 사용
BASELINE = 0.92  # 발 기준점 y (프레임 높이 비율)

UNIT_SPEC = {
    "ally": {"grid": (6, 6), "actions": ["attack", "defend", "skill", "move_slow", "move_fast", "crit"], "ref": "move_slow"},
    "enemy": {"grid": (5, 5), "actions": ["move", "attack", "defend", "crit", "hit"], "ref": "move"},
    "npc": {"grid": (5, 5), "actions": ["idle", "talk", "give"], "ref": "idle"},
}
UNIT_FOLDER = {"ally": "chars", "enemy": "enemies", "npc": "npcs"}
# 태그 → (프레임 크기, 기준 치수, 목표 px). 보병 몸체 62px = 화면 높이의 약 1/9 (×2 배율 기준)
UNIT_TARGET = {"S": (100, "h", 62), "B": (200, "h", 112), "L": (200, "w", 140)}
NPC_TARGET = (100, "h", 60)

PALETTE = {"unit": 48, "tile": 32, "prop": 48, "fx": 64, "icon": 24, "card": 48, "overlay": 32, "map": 64}
SOFT_PROPS = {"burnt_truck", "drum_fire"}  # 연기·불꽃이 있어 알파 5단계 유지
ALPHA_STEPS = np.array([0, 64, 128, 191, 255], dtype=np.int16)


# ------------------------------------------------------------------ 기본 연산

def load_rgba(path: str) -> np.ndarray:
    with Image.open(path) as im:
        return np.array(im.convert("RGBA"))


def save_rgba(arr: np.ndarray, path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp.png"
    Image.fromarray(arr, "RGBA").save(tmp, optimize=True)
    os.replace(tmp, path)


def ensure_alpha(arr: np.ndarray) -> tuple[np.ndarray, bool]:
    """투명 픽셀이 거의 없으면 테두리 지배색(최대 2색, 체크무늬 대비)을 투명 처리한다."""
    a = arr[..., 3]
    if (a < 250).mean() > 0.02:
        return arr, False
    h, w = a.shape
    k = max(2, min(h, w) // 128)
    border = np.concatenate([
        arr[:k, :, :3].reshape(-1, 3), arr[-k:, :, :3].reshape(-1, 3),
        arr[:, :k, :3].reshape(-1, 3), arr[:, -k:, :3].reshape(-1, 3),
    ]).astype(np.int32)
    q = border // 16
    keys, counts = np.unique(q[:, 0] * 256 + q[:, 1] * 16 + q[:, 2], return_counts=True)
    order = np.argsort(-counts)
    out = arr.copy()
    rgb = arr[..., :3].astype(np.int32)
    keyed = False
    for idx in order[:2]:
        if counts[idx] / len(border) < 0.15:
            break
        sel = border[(q[:, 0] * 256 + q[:, 1] * 16 + q[:, 2]) == keys[idx]]
        color = np.median(sel, axis=0)
        dist = np.sqrt(((rgb - color) ** 2).sum(axis=-1))
        out[dist < 48, 3] = 0
        keyed = True
    return out, keyed


def cell_boxes(w: int, h: int, cols: int, rows: int) -> list[list[tuple[int, int, int, int]]]:
    xs = [round(i * w / cols) for i in range(cols + 1)]
    ys = [round(j * h / rows) for j in range(rows + 1)]
    return [[(xs[c], ys[r], xs[c + 1], ys[r + 1]) for c in range(cols)] for r in range(rows)]


def crop(arr: np.ndarray, box: tuple[int, int, int, int]) -> np.ndarray:
    x0, y0, x1, y1 = box
    return arr[y0:y1, x0:x1]


def _best_cut(profile: np.ndarray, nominal: float, window: float) -> int:
    """nominal 근처(±window)에서 불투명 픽셀이 가장 적은 줄. 동률이면 nominal에 가장 가까운 줄"""
    lo = max(1, int(nominal - window))
    hi = min(len(profile) - 1, int(nominal + window))
    seg = profile[lo:hi + 1]
    cand = np.nonzero(seg == seg.min())[0] + lo
    return int(cand[np.argmin(np.abs(cand - nominal))])


def smart_boxes(arr: np.ndarray, cols: int, rows: int, search: float = 0.3):
    """그림 사이 빈 곳을 따라 자른 격자.
    AI는 행·열을 정확한 균등 간격으로 그리지 않아서(조금씩 밀림) 균등 분할로 자르면 이웃 칸의 발끝·총구가 잘려 들어온다.
    균등 분할선 근처(칸 크기의 ±search)에서 그림이 가장 적은 줄을 골라 자른다. 열은 행마다 따로 찾는다."""
    h, w = arr.shape[:2]
    opaque = arr[..., 3] >= 64
    ch, cw = h / rows, w / cols
    rprof = opaque.sum(axis=1)
    ys = [0] + [_best_cut(rprof, k * ch, ch * search) for k in range(1, rows)] + [h]
    boxes = []
    for r in range(rows):
        cprof = opaque[ys[r]:ys[r + 1]].sum(axis=0)
        xs = [0] + [_best_cut(cprof, k * cw, cw * search) for k in range(1, cols)] + [w]
        boxes.append([(xs[c], ys[r], xs[c + 1], ys[r + 1]) for c in range(cols)])
    return boxes


def components(mask: np.ndarray):
    """8-연결 성분 라벨링(런 길이 기반, 의존성 없음). 성분별 {area, box(x0,y0,x1,y1), runs[(y,s,e)]}"""
    parent: list[int] = []
    runs: list[tuple[int, int, int]] = []

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    prev: list[int] = []
    m8 = mask.astype(np.int8)
    for y in range(mask.shape[0]):
        if not mask[y].any():
            prev = []
            continue
        d = np.diff(np.concatenate(([0], m8[y], [0])))
        cur = []
        for st, en in zip(np.nonzero(d == 1)[0], np.nonzero(d == -1)[0]):
            idx = len(runs)
            runs.append((y, int(st), int(en)))
            parent.append(idx)
            cur.append(idx)
            for p in prev:
                _, ps, pe = runs[p]
                if ps <= en and pe >= st:  # 대각선 포함 인접
                    ra, rb = find(idx), find(p)
                    if ra != rb:
                        parent[ra] = rb
        prev = cur
    comps: dict[int, dict] = {}
    for i, (y, st, en) in enumerate(runs):
        c = comps.setdefault(find(i), {"area": 0, "box": [st, y, en, y + 1], "runs": []})
        c["area"] += en - st
        b = c["box"]
        b[0], b[1], b[2], b[3] = min(b[0], st), min(b[1], y), max(b[2], en), max(b[3], y + 1)
        c["runs"].append((y, st, en))
    return list(comps.values())


def remove_edge_fragments(cell: np.ndarray, ratio: float = 0.15) -> tuple[np.ndarray, int]:
    """칸 테두리에 닿아 있고 가장 큰 덩어리의 ratio 미만인 조각(이웃 칸에서 넘어온 것)을 지운다.
    테두리에 닿지 않는 작은 조각(먼지·섬광·탄피)은 남긴다."""
    comps = components(cell[..., 3] >= 64)
    if len(comps) < 2:
        return cell, 0
    h, w = cell.shape[:2]
    largest = max(c["area"] for c in comps)
    out, removed = cell, 0
    for c in comps:
        x0, y0, x1, y1 = c["box"]
        if c["area"] < largest * ratio and (x0 == 0 or y0 == 0 or x1 == w or y1 == h):
            if removed == 0:
                out = cell.copy()
            for y, st, en in c["runs"]:
                out[y, st:en] = 0
            # 반투명 가장자리(알파 64 미만)도 같이 지운다
            out[max(0, y0 - 1):min(h, y1 + 1), max(0, x0 - 1):min(w, x1 + 1)][out[max(0, y0 - 1):min(h, y1 + 1), max(0, x0 - 1):min(w, x1 + 1), 3] < 64] = 0
            removed += 1
    return out, removed


def extract_cells(arr: np.ndarray, cols: int, rows: int, search: float = 0.3, clean: bool = True):
    """smart_boxes로 자르고 조각을 정리한 칸들을, 균등 격자 기준 좌표계를 유지한 같은 크기 캔버스에 담아 돌려준다.
    (자르는 선은 칸마다 달라도 좌표 원점은 균등 격자 그대로여야 프레임 간 위치·발 기준선이 흔들리지 않는다)"""
    h, w = arr.shape[:2]
    ch, cw = h / rows, w / cols
    pw, ph = int(math.ceil(cw * search)) + 1, int(math.ceil(ch * search)) + 1
    W, H = int(math.ceil(cw)) + 2 * pw, int(math.ceil(ch)) + 2 * ph
    boxes = smart_boxes(arr, cols, rows, search)
    grid, removed = [], 0
    for r in range(rows):
        row = []
        for c in range(cols):
            x0, y0, x1, y1 = boxes[r][c]
            part = arr[y0:y1, x0:x1]
            if clean:
                part, n = remove_edge_fragments(part)
                removed += n
            ox, oy = int(round(c * cw)) - pw, int(round(r * ch)) - ph
            cell = np.zeros((H, W, 4), np.uint8)
            dx0, dy0 = max(0, x0 - ox), max(0, y0 - oy)
            dx1, dy1 = min(W, x1 - ox), min(H, y1 - oy)
            cell[dy0:dy1, dx0:dx1] = part[dy0 - (y0 - oy):dy1 - (y0 - oy), dx0 - (x0 - ox):dx1 - (x0 - ox)]
            row.append(cell)
        grid.append(row)
    return grid, removed


def bbox(arr: np.ndarray, thr: int = 128):
    ys, xs = np.nonzero(arr[..., 3] >= thr)
    if len(xs) == 0:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1


def union(boxes):
    boxes = [b for b in boxes if b]
    if not boxes:
        return None
    return min(b[0] for b in boxes), min(b[1] for b in boxes), max(b[2] for b in boxes), max(b[3] for b in boxes)


def resize(arr: np.ndarray, w: int, h: int) -> np.ndarray:
    """premultiplied 알파로 크기 변경(테두리 검은 번짐 방지). 축소 BOX, 확대 NEAREST."""
    w, h = max(1, int(round(w))), max(1, int(round(h)))
    if (w, h) == (arr.shape[1], arr.shape[0]):
        return arr.copy()
    shrink = w < arr.shape[1] or h < arr.shape[0]
    im = Image.fromarray(arr, "RGBA").convert("RGBa")
    im = im.resize((w, h), Image.Resampling.BOX if shrink else Image.Resampling.NEAREST)
    return np.array(im.convert("RGBA"))


def paste(dst: np.ndarray, src: np.ndarray, x: int, y: int) -> None:
    """알파 합성으로 붙이기(범위 밖은 잘림)."""
    H, W = dst.shape[:2]
    h, w = src.shape[:2]
    x0, y0 = max(0, x), max(0, y)
    x1, y1 = min(W, x + w), min(H, y + h)
    if x0 >= x1 or y0 >= y1:
        return
    s = src[y0 - y:y1 - y, x0 - x:x1 - x].astype(np.float32)
    d = dst[y0:y1, x0:x1].astype(np.float32)
    sa = s[..., 3:4] / 255.0
    da = d[..., 3:4] / 255.0
    oa = sa + da * (1 - sa)
    rgb = np.where(oa > 0, (s[..., :3] * sa + d[..., :3] * da * (1 - sa)) / np.maximum(oa, 1e-6), 0)
    dst[y0:y1, x0:x1, :3] = np.clip(rgb, 0, 255).astype(np.uint8)
    dst[y0:y1, x0:x1, 3] = np.clip(oa[..., 0] * 255, 0, 255).astype(np.uint8)


def harden_alpha(arr: np.ndarray, mode: str) -> np.ndarray:
    out = arr.copy()
    a = out[..., 3].astype(np.int16)
    if mode == "hard":
        a = np.where(a >= 128, 255, 0)
    elif mode == "steps":
        a = ALPHA_STEPS[np.abs(a[..., None] - ALPHA_STEPS[None, None, :]).argmin(axis=-1)]
    elif mode == "none":
        a = np.full_like(a, 255)
    out[..., 3] = a.astype(np.uint8)
    out[out[..., 3] == 0, :3] = 0
    return out


def quantize(arr: np.ndarray, colors: int) -> np.ndarray:
    """불투명 픽셀 색을 colors개 이하로(디더링 없음). 시트 전체에 한 팔레트."""
    mask = arr[..., 3] > 0
    if mask.sum() == 0:
        return arr
    rgb = arr[..., :3].copy()
    rgb[~mask] = rgb[mask].mean(axis=0).astype(np.uint8)
    q = Image.fromarray(rgb, "RGB").quantize(colors=colors, method=Image.Quantize.MEDIANCUT, dither=Image.Dither.NONE)
    out = arr.copy()
    out[..., :3] = np.where(mask[..., None], np.array(q.convert("RGB")), 0)
    return out


def to_sheet(frames: list[np.ndarray], fs: int) -> np.ndarray:
    sheet = np.zeros((fs * OUT_GRID, fs * OUT_GRID, 4), np.uint8)
    for i, f in enumerate(frames[: OUT_GRID * OUT_GRID]):
        r, c = divmod(i, OUT_GRID)
        sheet[r * fs:(r + 1) * fs, c * fs:(c + 1) * fs] = f
    return sheet


def fit_frames(cells: list[np.ndarray], fs: int, metric: str, target: float, anchor_bottom: bool, ref_cells=None, warn=None):
    """같은 배율·같은 기준점으로 여러 칸을 fs×fs 프레임에 배치한다."""
    ref = ref_cells if ref_cells is not None else cells
    rb = union([bbox(c) for c in ref]) or union([bbox(c) for c in cells])
    if rb is None:
        return [np.zeros((fs, fs, 4), np.uint8) for _ in cells], 1.0
    rw, rh = rb[2] - rb[0], rb[3] - rb[1]
    if metric == "contain":
        s = min(fs * target / rw, fs * target / rh)
    else:
        s = target / (rh if metric == "h" else rw)
    cx = (rb[0] + rb[2]) / 2
    ay_src = rb[3] if anchor_bottom else (rb[1] + rb[3]) / 2
    ax_dst = fs / 2
    ay_dst = fs * BASELINE if anchor_bottom else fs / 2
    # 모든 프레임이 프레임 안에 들어오는지 확인하고 넘치면 배율을 줄인다
    ab = union([bbox(c) for c in cells])
    if ab:
        need = [
            (cx - ab[0]) / ax_dst if cx > ab[0] else 0,
            (ab[2] - cx) / (fs - ax_dst) if ab[2] > cx else 0,
            (ay_src - ab[1]) / ay_dst if ay_src > ab[1] else 0,
            (ab[3] - ay_src) / (fs - ay_dst) if ab[3] > ay_src else 0,
        ]
        smax = 1 / max(max(need), 1e-6)
        if s > smax:
            if warn is not None and s > smax * 1.08:
                warn.append(f"프레임에 넘쳐 배율 {s:.3f}→{smax:.3f}로 축소(일부 프레임의 동작 범위가 큼)")
            s = smax
    out = []
    for c in cells:
        h, w = c.shape[:2]
        scaled = resize(c, w * s, h * s)
        frame = np.zeros((fs, fs, 4), np.uint8)
        paste(frame, scaled, int(round(ax_dst - cx * s)), int(round(ay_dst - ay_src * s)))
        out.append(frame)
    return out, s


# ------------------------------------------------------------------ 종류별 처리

def process_unit(img, post, out_root, report):
    spec = UNIT_SPEC[post["faction"]]
    cols, rows = spec["grid"]
    nframes = cols
    grid, removed = extract_cells(img, cols, rows)
    cells = {a: grid[r][:nframes] for r, a in enumerate(spec["actions"])}
    if removed:
        report["warnings"].append(f"이웃 칸에서 넘어온 조각 {removed}개 제거")
    for a, row in cells.items():
        empty = sum(bbox(c) is None for c in row)
        if empty:
            report["warnings"].append(f"{a}: 빈 프레임 {empty}개")

    fs, metric, target = NPC_TARGET if post["faction"] == "npc" else UNIT_TARGET[post["tag"]]
    all_cells = [c for row in cells.values() for c in row]
    frames, s = fit_frames(all_cells, fs, metric, target, True, ref_cells=cells[spec["ref"]], warn=report["warnings"])
    frames = [harden_alpha(f, "hard") for f in frames]
    # 유닛 전체에 한 팔레트: 프레임을 이어 붙여 양자화 후 다시 나눈다
    strip = quantize(np.concatenate(frames, axis=1), PALETTE["unit"])
    frames = [strip[:, i * fs:(i + 1) * fs] for i in range(len(frames))]
    report["scale"] = round(s, 4)

    suffix = "_L" if fs > 100 else ""
    outs = []
    for r, a in enumerate(spec["actions"]):
        sheet = to_sheet(frames[r * nframes:(r + 1) * nframes], fs)
        rel = f"{UNIT_FOLDER[post['faction']]}/{post['id']}_{a}{suffix}.png"
        save_rgba(sheet, os.path.join(out_root, rel))
        outs.append((rel, frames[r * nframes:(r + 1) * nframes]))
    return outs


def process_tiles(img, post, out_root, report):
    img = harden_alpha(img, "none")
    boxes = cell_boxes(img.shape[1], img.shape[0], 2, 2)
    outs = []
    for i, tid in enumerate(post["ids"]):
        r, c = divmod(i, 2)
        tile = quantize(resize(crop(img, boxes[r][c]), 64, 64), PALETTE["tile"])
        rgb = tile[..., :3].astype(np.int16)
        inner = np.abs(np.diff(rgb, axis=1)).mean() + 1e-6
        seam = max(np.abs(rgb[:, 0] - rgb[:, -1]).mean(), np.abs(rgb[0] - rgb[-1]).mean())
        if seam > inner * 2.5:
            report["warnings"].append(f"{tid}: 이음매가 보일 수 있음(경계 차이 {seam:.0f} vs 내부 {inner:.0f})")
        rel = f"tiles/{tid}.png"
        save_rgba(tile, os.path.join(out_root, rel))
        outs.append((rel, [np.tile(tile, (2, 2, 1))]))
    return outs


def process_strips(img, post, out_root, report, kind):
    """props·fx: 6열 x 5행, 행마다 하나의 6프레임 애니메이션"""
    grid, removed = extract_cells(img, 6, 5)
    if removed:
        report["warnings"].append(f"이웃 칸에서 넘어온 조각 {removed}개 제거")
    outs = []
    for r, row in enumerate(post["rows"]):
        cells = grid[r]
        if all(bbox(c, 32) is None for c in cells):
            report["warnings"].append(f"{row['name']}: 행이 비어 있음")
            continue
        fs = row["frame"]
        if kind == "prop":
            frames, _ = fit_frames(cells, fs, row["fit"], row["px"], True, warn=report["warnings"])
            mode = "steps" if row["name"] in SOFT_PROPS else "hard"
            folder = "props"
        else:
            frames, _ = fit_frames(cells, fs, "contain", 0.96, False, warn=report["warnings"])
            mode = "steps"
            folder = "fx"
        frames = [harden_alpha(f, mode) for f in frames]
        strip = quantize(np.concatenate(frames, axis=1), PALETTE[kind])
        frames = [strip[:, i * fs:(i + 1) * fs] for i in range(6)]
        suffix = "_L" if kind == "prop" and fs > 100 else ""
        rel = f"{folder}/{row['name']}{suffix}.png"
        save_rgba(to_sheet(frames, fs), os.path.join(out_root, rel))
        outs.append((rel, frames))
    return outs


def _cover(cell, w, h):
    ch, cw = cell.shape[:2]
    s = max(w / cw, h / ch)
    big = resize(cell, cw * s, ch * s)
    y0 = (big.shape[0] - h) // 2
    x0 = (big.shape[1] - w) // 2
    return big[y0:y0 + h, x0:x0 + w]


def process_grid(img, post, out_root, report):
    """아이콘·오버레이·카드 일러스트: cols x rows 격자, 칸마다 하나의 이미지"""
    w, h = post["w"], post["h"]
    mode = post.get("alpha", "hard")
    fill = post.get("fill", "contain")
    # 꽉 찬 그림(일러스트·오버레이)은 균등 분할, 투명 배경 아이콘은 빈 곳을 따라 자르고 조각 정리
    boxes = cell_boxes(img.shape[1], img.shape[0], post["cols"], post["rows"])
    smart = None
    if fill != "stretch":
        smart, removed = extract_cells(img, post["cols"], post["rows"])
        if removed:
            report["warnings"].append(f"이웃 칸에서 넘어온 조각 {removed}개 제거")
    pal = PALETTE["card"] if mode == "none" else PALETTE["overlay"] if mode == "steps" else PALETTE["icon"]
    outs = []
    for i, name in enumerate(post["names"]):
        r, c = divmod(i, post["cols"])
        cell = smart[r][c] if smart else crop(img, boxes[r][c])
        if fill == "stretch":
            m = max(1, int(min(cell.shape[:2]) * 0.02))  # 이웃 칸이 번진 가장자리 제거
            out = _cover(cell[m:-m, m:-m], w, h)
        else:
            b = bbox(cell, 32)
            if b is None:
                report["warnings"].append(f"{name}: 빈 칸")
                continue
            part = cell[b[1]:b[3], b[0]:b[2]]
            s = min((w - 2) / part.shape[1], (h - 2) / part.shape[0])
            out = np.zeros((h, w, 4), np.uint8)
            small = resize(part, part.shape[1] * s, part.shape[0] * s)
            paste(out, small, (w - small.shape[1]) // 2, (h - small.shape[0]) // 2)
        out = quantize(harden_alpha(out, mode), pal)
        rel = f"{post['folder']}/{name}.png"
        save_rgba(out, os.path.join(out_root, rel))
        outs.append((rel, [out]))
    return outs


def process_row(img, post, out_root, report):
    """카드 프레임 5장: 투명 열 간격으로 나누고(실패 시 균등 분할) 각 카드를 w×h로"""
    n, w, h = post["count"], post["w"], post["h"]
    colmask = (img[..., 3] >= 64).any(axis=0)
    segs, start = [], None
    for x, on in enumerate(colmask):
        if on and start is None:
            start = x
        elif not on and start is not None:
            segs.append((start, x))
            start = None
    if start is not None:
        segs.append((start, len(colmask)))
    segs = [s for s in segs if s[1] - s[0] > img.shape[1] / (n * 4)]
    if len(segs) != n:
        report["warnings"].append(f"카드 {n}장을 간격으로 구분하지 못해 균등 분할({len(segs)}개 감지)")
        segs = [(round(i * img.shape[1] / n), round((i + 1) * img.shape[1] / n)) for i in range(n)]
    outs = []
    for (x0, x1), name in zip(segs, post["names"]):
        part = img[:, x0:x1]
        b = bbox(part, 64)
        if b is None:
            report["warnings"].append(f"{name}: 비어 있음")
            continue
        card = resize(part[b[1]:b[3], b[0]:b[2]], w, h)
        card = quantize(harden_alpha(card, "steps"), PALETTE["card"])
        rel = f"{post['folder']}/{name}.png"
        save_rgba(card, os.path.join(out_root, rel))
        outs.append((rel, [card]))
    return outs


def process_map(img, post, out_root, report):
    out = quantize(harden_alpha(resize(img, post["w"], post["h"]), "none"), PALETTE["map"])
    rel = "map/korea_map.png"
    save_rgba(out, os.path.join(out_root, rel))
    return [(rel, [out])]


PROCESSORS = {
    "unit": process_unit,
    "tiles": process_tiles,
    "props": lambda i, p, o, r: process_strips(i, p, o, r, "prop"),
    "fx": lambda i, p, o, r: process_strips(i, p, o, r, "fx"),
    "grid": process_grid,
    "row": process_row,
    "map": process_map,
}


# ------------------------------------------------------------------ 미리보기

def checker(h, w, size=8):
    yy, xx = np.mgrid[0:h, 0:w]
    v = np.where(((yy // size) + (xx // size)) % 2 == 0, 70, 90).astype(np.uint8)
    return np.dstack([v, v, v, np.full_like(v, 255)])


def build_preview(outs, scale=2, max_w=1600) -> np.ndarray:
    """출력 프레임들을 체크무늬 위에 2배로 늘어놓은 확인용 이미지"""
    rows = []
    for _rel, frames in outs:
        tiles = [np.repeat(np.repeat(f, scale, 0), scale, 1) for f in frames]
        h = max(t.shape[0] for t in tiles)
        line = []
        for t in tiles:
            pad = np.zeros((h, t.shape[1] + 4, 4), np.uint8)
            pad[: t.shape[0], : t.shape[1]] = t
            line.append(pad)
        row = np.concatenate(line, axis=1)
        if row.shape[1] > max_w:
            row = row[:, :max_w]
        rows.append(row)
    W = max(r.shape[1] for r in rows)
    H = sum(r.shape[0] + 4 for r in rows)
    canvas = checker(H, W)
    y = 0
    for r in rows:
        paste(canvas, r, 0, y)
        y += r.shape[0] + 4
    return canvas


# ------------------------------------------------------------------ 진입점

def process_item(item: dict, raw_path: str, out_dir: str) -> dict:
    report = {"name": item["name"], "ok": False, "outputs": [], "warnings": []}
    try:
        img = load_rgba(raw_path)
        if item.get("transparent", True) and item["post"]["kind"] not in ("tiles", "map"):
            img, keyed = ensure_alpha(img)
            if keyed:
                report["warnings"].append("원본에 투명 배경이 없어 배경색을 제거함")
        out_root = os.path.join(out_dir, "public", "assets")
        outs = PROCESSORS[item["post"]["kind"]](img, item["post"], out_root, report)
        report["outputs"] = [rel for rel, _ in outs]
        if outs:
            save_rgba(build_preview(outs), os.path.join(out_dir, "preview", f"{item['name']}.png"))
        report["ok"] = bool(outs)
    except Exception as e:  # 한 장의 실패가 전체를 멈추지 않게
        report["warnings"].append(f"처리 실패: {type(e).__name__}: {e}")
    return report


def process_all(items: list[dict], raw_dir: str, out_dir: str, only=None) -> list[dict]:
    reports = []
    for item in items:
        if only and item["name"] not in only:
            continue
        raw = os.path.join(raw_dir, item["name"] + ".png")
        if not os.path.exists(raw):
            continue
        rep = process_item(item, raw, out_dir)
        reports.append(rep)
        flag = "OK " if rep["ok"] else "ERR"
        print(f"[{flag}] {item['name']}: 출력 {len(rep['outputs'])}개" + (f" / 경고 {len(rep['warnings'])}" if rep["warnings"] else ""))
        for w in rep["warnings"]:
            print("      -", w)
    with open(os.path.join(out_dir, "report.json"), "w", encoding="utf-8") as f:
        json.dump(reports, f, ensure_ascii=False, indent=1)
    return reports


def main(argv=None):
    here = os.path.dirname(os.path.abspath(__file__))
    ap = argparse.ArgumentParser(description="AI 원본 이미지를 게임 규격 에셋으로 변환")
    ap.add_argument("--raw", required=True, help="원본 폴더(<항목 이름>.png)")
    ap.add_argument("--out", required=True, help="출력 폴더(public/assets, preview, report.json 생성)")
    ap.add_argument("--prompts", default=os.path.join(here, "prompts.json"))
    ap.add_argument("--only", default="", help="쉼표로 구분한 항목 이름")
    args = ap.parse_args(argv)
    with open(args.prompts, encoding="utf-8") as f:
        items = json.load(f)
    only = {s.strip() for s in args.only.split(",") if s.strip()} or None
    reps = process_all(items, args.raw, args.out, only)
    bad = [r for r in reps if not r["ok"]]
    print(f"\n처리 {len(reps)}개, 실패 {len(bad)}개 → {os.path.join(args.out, 'public', 'assets')}")
    return 1 if bad else 0


In [ ]:
#@title 5) 모델 확인과 작업 목록 { display-mode: "form" }
PREFERRED = ["gpt-image-2.5-flare", "gpt-image-2.5", "gpt-image-1.5", "gpt-image-1"]

def list_image_models():
    try:
        r = requests.get("https://api.openai.com/v1/models", headers=HEADERS, timeout=30)
        if r.status_code == 401:
            raise SystemExit("API 키가 올바르지 않습니다(401). 키를 다시 확인하세요.")
        r.raise_for_status()
        return sorted(m["id"] for m in r.json().get("data", []) if "image" in m["id"])
    except SystemExit:
        raise
    except Exception as e:
        print("모델 목록을 가져오지 못했습니다:", e)
        return []

available = [] if CONVERT_ONLY else list_image_models()
if not CONVERT_ONLY:
    print("이 계정의 이미지 모델:", ", ".join(available) or "(확인 불가)")
if MODEL != "auto":
    USE_MODEL = MODEL
    if available and MODEL not in available:
        print(f"경고: {MODEL}이(가) 목록에 없습니다. 그대로 시도합니다.")
else:
    USE_MODEL = next((m for m in PREFERRED if m in available), None) or PREFERRED[0]
print("사용할 모델:", USE_MODEL)

def prompt_hash(item):
    return hashlib.sha256(json.dumps([item["prompt"], item["size"], item["transparent"]]).encode()).hexdigest()[:16]

def raw_path(item):
    return os.path.join(RAW_DIR, item["name"] + ".png")

def meta_path(item):
    return os.path.join(RAW_DIR, item["name"] + ".json")

def is_changed(item):
    try:
        with open(meta_path(item), encoding="utf-8") as f:
            return json.load(f).get("prompt_hash") != prompt_hash(item)
    except Exception:
        return False

only = {s.strip() for s in ONLY.split(",") if s.strip()}
regen = {s.strip() for s in REGENERATE.split(",") if s.strip()}
selected = [it for it in PROMPTS if (it["name"] in only if only else (PHASE == "ALL" or it["phase"] == PHASE))]
unknown = (only | (regen - {"changed"})) - {it["name"] for it in PROMPTS}
if unknown:
    print("알 수 없는 항목 이름:", ", ".join(sorted(unknown)))

todo = []
for it in selected:
    exists = os.path.exists(raw_path(it))
    redo = it["name"] in regen or ("changed" in regen and is_changed(it))
    if not exists or redo:
        todo.append(it)
if LIMIT > 0:
    todo = todo[:LIMIT]
if CONVERT_ONLY:
    todo = []
    print("변환만 모드: 새 그림은 만들지 않습니다.")

# 대략적인 비용(장당 달러, 1024x1024 기준). 실제 청구액은 OpenAI 사용량 페이지에서 확인
PRICE = {"low": 0.006, "medium": 0.013, "high": 0.053} if "2.5" in USE_MODEL else {"low": 0.011, "medium": 0.042, "high": 0.167}
est = sum(PRICE[QUALITY] * (1.5 if it["size"] != "1024x1024" else 1) for it in todo)
done_count = sum(os.path.exists(raw_path(it)) for it in selected)
print(f"선택 {len(selected)}개 중 완료 {done_count}개 / 이번에 만들 것 {len(todo)}개, 예상 약 ${est:.2f} ({QUALITY})")
for it in todo[:20]:
    print("  -", it["no"], it["name"])
if len(todo) > 20:
    print(f"  ... 외 {len(todo) - 20}개")


In [ ]:
#@title 6) 이미지 만들기 { display-mode: "form" }
class Fatal(Exception):
    pass

print_lock = threading.Lock()
stop = threading.Event()
dropped_params = set()  # 모델이 지원하지 않는 선택 파라미터(한 번 확인되면 모두에 적용)

def log(*a):
    with print_lock:
        print(*a, flush=True)

def make_body(item):
    body = {"model": USE_MODEL, "prompt": item["prompt"], "size": item["size"], "quality": QUALITY,
            "n": 1, "output_format": "png"}
    if item["transparent"]:
        body["background"] = "transparent"
    if MODERATION == "low":
        body["moderation"] = "low"
    for p in dropped_params:
        body.pop(p, None)
    return body

def request_image(item):
    attempt = 0
    while not stop.is_set():
        attempt += 1
        body = make_body(item)
        try:
            r = requests.post("https://api.openai.com/v1/images/generations", headers=HEADERS, json=body, timeout=600)
        except requests.RequestException as e:
            if attempt >= 5:
                raise RuntimeError(f"네트워크 오류 반복: {e}")
            time.sleep(min(60, 5 * 2 ** attempt) + random.random() * 3)
            continue
        if r.status_code == 200:
            return r.json()
        msg = r.text[:600]
        try:
            err = r.json().get("error", {})
            msg = err.get("message", msg)
            code = err.get("code") or ""
            param = err.get("param") or ""
        except Exception:
            code = param = ""
        low = msg.lower()
        if r.status_code in (401, 403):
            hint = " (조직 인증이 필요할 수 있습니다: platform.openai.com → Settings → Organization → Verify)" if "verif" in low else ""
            raise Fatal(f"{r.status_code} 권한 오류: {msg}{hint}")
        if r.status_code == 404:
            raise Fatal(f"모델을 찾을 수 없습니다({USE_MODEL}): {msg}")
        if r.status_code == 429 and ("quota" in low or "billing" in low or code == "insufficient_quota"):
            raise Fatal("크레딧이 부족합니다. platform.openai.com → Billing에서 충전하세요.")
        if r.status_code == 400:
            optional = {"moderation", "background", "output_format", "quality"}
            bad = param if param in optional else next((p for p in optional if p in low and ("unknown" in low or "unsupported" in low or "invalid" in low)), None)
            if bad and bad not in dropped_params and bad in body:
                dropped_params.add(bad)
                log(f"  ({USE_MODEL}가 '{bad}' 파라미터를 지원하지 않아 빼고 다시 시도)")
                continue
            if "safety" in low or "moderation" in low or code in ("moderation_blocked", "content_policy_violation"):
                raise RuntimeError(f"안전 필터에 막힘: {msg[:200]} → MODERATION=low로 다시 시도하거나 프롬프트 수정")
            raise RuntimeError(f"요청 오류 400: {msg[:300]}")
        if r.status_code == 429 or r.status_code >= 500:
            if attempt >= 6:
                raise RuntimeError(f"{r.status_code} 반복: {msg[:200]}")
            wait = float(r.headers.get("retry-after", 0) or 0) or min(90, 5 * 2 ** attempt)
            log(f"  {item['name']}: {r.status_code} 대기 {wait:.0f}초 후 재시도 ({attempt}/6)")
            time.sleep(wait + random.random() * 2)
            continue
        raise RuntimeError(f"{r.status_code}: {msg[:300]}")
    raise RuntimeError("중단됨")

def generate(item):
    t0 = time.time()
    data = request_image(item)
    d = data["data"][0]
    png = base64.b64decode(d["b64_json"]) if d.get("b64_json") else requests.get(d["url"], timeout=120).content
    from PIL import Image
    import io
    with Image.open(io.BytesIO(png)) as im:
        im.load()
        has_alpha = im.mode in ("RGBA", "LA") and im.getextrema()[-1][0] < 250
        size = f"{im.width}x{im.height}"
    tmp = raw_path(item) + ".tmp"
    with open(tmp, "wb") as f:
        f.write(png)
    os.replace(tmp, raw_path(item))  # 중간에 끊겨도 깨진 파일이 '완료'로 남지 않게
    meta = {"name": item["name"], "prompt_hash": prompt_hash(item), "model": USE_MODEL, "quality": QUALITY,
            "size": size, "transparent_requested": item["transparent"], "has_alpha": has_alpha,
            "seconds": round(time.time() - t0, 1), "usage": data.get("usage"), "created": time.strftime("%Y-%m-%d %H:%M:%S")}
    with open(meta_path(item), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=1)
    note = "" if (has_alpha or not item["transparent"]) else " (투명 배경 아님 → 후처리에서 배경 제거)"
    return f"{meta['seconds']}초{note}"

made, failed = [], []
if todo:
    workers = max(1, min(CONCURRENCY, len(todo)))
    log(f"{len(todo)}장 생성 시작 (모델 {USE_MODEL}, 품질 {QUALITY}, 동시 {workers})")
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = {ex.submit(generate, it): it for it in todo}
        for i, fut in enumerate(as_completed(futs), 1):
            it = futs[fut]
            try:
                log(f"[{i}/{len(todo)}] 완료 {it['no']} {it['name']} - {fut.result()}")
                made.append(it["name"])
            except Fatal as e:
                stop.set()
                log(f"[중단] {e}")
                failed.append((it["name"], str(e)))
            except Exception as e:
                log(f"[{i}/{len(todo)}] 실패 {it['no']} {it['name']} - {e}")
                failed.append((it["name"], str(e)))
elif not CONVERT_ONLY:
    log("새로 만들 그림이 없습니다. (LIMIT, PHASE, ONLY, REGENERATE 설정 확인)")
log(f"\n이번 실행: 성공 {len(made)}장, 실패 {len(failed)}장")
for name, err in failed:
    log(f"  - {name}: {err[:200]}")


In [ ]:
#@title 7) 게임 규격으로 변환 + 미리보기 { display-mode: "form" }
try:
    from IPython.display import Image as IPyImage, display
except ImportError:  # Colab 밖(일반 파이썬)에서는 미리보기 표시 생략
    IPyImage = display = None
targets = {it["name"] for it in selected if os.path.exists(raw_path(it))}
reports = process_all([it for it in PROMPTS if it["name"] in targets], RAW_DIR, OUT_DIR)
shown = 0
for rep in reports:
    if rep["name"] in made or (not made and shown < 6):
        p = os.path.join(OUT_DIR, "preview", rep["name"] + ".png")
        if display and os.path.exists(p) and shown < 12:
            print("미리보기:", rep["name"])
            display(IPyImage(filename=p))
            shown += 1
warn = sum(1 for r in reports if r["warnings"])
print(f"변환 {len(reports)}개 (경고 있는 항목 {warn}개). 전체 미리보기: {os.path.join(OUT_DIR, 'preview')}")


In [ ]:
#@title 8) 묶어서 내려받기 { display-mode: "form" }
def zip_dir(src, zip_path, arc_root):
    n = 0
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(src):
            for fn in sorted(files):
                if fn.endswith(".tmp") or fn.endswith(".tmp.png"):
                    continue
                full = os.path.join(root, fn)
                z.write(full, os.path.join(arc_root, os.path.relpath(full, src)))
                n += 1
    return n

zips = []
n = zip_dir(os.path.join(OUT_DIR, "public"), os.path.join(BASE, "game_assets.zip"), "public")
zips.append(os.path.join(BASE, "game_assets.zip"))
print(f"game_assets.zip: 파일 {n}개 (압축을 풀어 저장소 루트에 올리면 public/assets에 들어갑니다)")
if ZIP_RAW:
    n = zip_dir(RAW_DIR, os.path.join(BASE, "raw_images.zip"), "raw")
    zips.append(os.path.join(BASE, "raw_images.zip"))
    print(f"raw_images.zip: 파일 {n}개")
if IN_COLAB:
    from google.colab import files
    for zp in zips:
        try:
            files.download(zp)
        except Exception as e:
            print("자동 다운로드가 안 되면 왼쪽 폴더 아이콘(또는 Drive)에서 직접 받으세요:", zp, e)
